# 00. Data Collection

data-analysis와 ai-model이 사용하는 원천 데이터를 수집합니다. 과거 `analysis-modeling/06_external_data_collection`의 운영 자동수집 섹션을 현재 레포 구조에 맞게 옮긴 노트북입니다.


In [ ]:
# ============================================================
# 00 공통 경로 설정
# ============================================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('[INFO] Google Colab 환경이 아니므로 drive.mount를 건너뜁니다.')

from pathlib import Path
import os

ROOT_PATH = "/content/drive/MyDrive/Data_analysis/The appropriateness of domestic oil prices compared to international oil prices/산업부/"
DATA_COLLECTION_PATH = os.path.join(ROOT_PATH, "data-analysis", "00_data_collection", "outputs") + "/"
DATA_PATH = os.path.join(DATA_COLLECTION_PATH, "_legacy_flat") + "/"
PROCESSED_PATH = os.path.join(ROOT_PATH, "preprocessed_data") + "/"

Path(DATA_COLLECTION_PATH).mkdir(parents=True, exist_ok=True)
Path(DATA_PATH).mkdir(parents=True, exist_ok=True)
Path(PROCESSED_PATH).mkdir(parents=True, exist_ok=True)

print(f"ROOT_PATH           = {ROOT_PATH}")
print(f"DATA_COLLECTION_PATH= {DATA_COLLECTION_PATH}")
print(f"DATA_PATH           = {DATA_PATH}")
print(f"PROCESSED_PATH      = {PROCESSED_PATH}")


## B. 운영 자동수집 검토/실행 셀

- B-0 공통 설정 셀
- fx_usdkrw.csv 환율
- crude.csv 원유가
- intl_products.zip, intl_product_diesel(0.001).csv 국제 석유제품
- retail_avg.csv 전국 평균 소매가
- brand_gasoline.csv, brand_diesel.csv 상표별 가격
- gasoline_tax_trend.xls, diesel_tax_trend.xls 유류세 변동
- refinery_weekly_supply_prices_by_product.csv 정유사 주간 공급가격
- korea_fuel_tax_price_policies.csv 정책 기준표
- 오피넷 지역별 주유소 원본 수집 → gasoline.csv, diesel.csv, metadata.json
- 대한석유협회 시설 목록 + 좌표 → facility_data.csv, 1 - facility_location_data_final.csv
- 주유소 주소 좌표 보강 → metadata__latlon.json
- 공시지가.csv 공시지가

해당 폴더에 각 작업 수행하는 폴더 만들어서 아래와 같이 진행한 후 결과 저장하면 됨.

1. 지금 사용하는 데이터 출처 파악
2. 출처에서 api 사용 가능한지 파악
3. api 사용가능하면 api 키 받는 방법 채팅으로 알려주기. api 안되면 크롤링 준비  
4. 데이터 수집 -> 함수화. 추후 웹사이트 자동화를 위해 그대로 함수 복사해서 들고 올거임. 각 이전 날짜 데이터 수집을 자동화 할껀데, 지금 단계는 기간으로 수집해야함. 즉, 단일/ 기간 모두 다 되게.
5. 데이터 전처리 방식 찾기 (이전 스텝에서 한거)
6. 데이터 전처리 코드
7.  최종 데이터 파일 저장


자료 조사를 많이 해야한다. 철저하게 진행하도록

### 설정

In [ ]:
# ============================================================
# B-0. 운영 자동수집 공통 설정
# ============================================================

from pathlib import Path
from datetime import datetime
import os
import json
import time
import zipfile
import shutil
import requests
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 수집 기간
# ------------------------------------------------------------
COLLECT_START_DATE = os.getenv("KFF_COLLECT_START_DATE", "2008-01-01")
COLLECT_END_DATE = os.getenv("KFF_COLLECT_END_DATE", pd.Timestamp.today(tz="Asia/Seoul").strftime("%Y-%m-%d"))

# ------------------------------------------------------------
# 실행 제어
# ------------------------------------------------------------
RUN_COLLECTION = os.getenv("KFF_RUN_COLLECTION", "true").lower() == "true"

# 각 데이터 셀에서 개별 실행 여부를 따로 켤 수 있게 둔다.
RUN_FX = True
RUN_CRUDE = True
RUN_INTL_PRODUCTS = True
RUN_RETAIL_AVG = True
RUN_BRAND_PRICE = True
RUN_TAX_TREND = True
RUN_REFINERY_WEEKLY = True
RUN_POLICY_TABLE = True
RUN_OPINET_STATION_DOWNLOAD = os.getenv("KFF_RUN_STATION_DOWNLOAD", "false").lower() == "true"
RUN_FACILITY = os.getenv("KFF_RUN_FACILITY", "false").lower() == "true"
RUN_METADATA_GEOCODE = os.getenv("KFF_RUN_METADATA_GEOCODE", "false").lower() == "true"
RUN_LAND_PRICE = os.getenv("KFF_RUN_LAND_PRICE", "false").lower() == "true"

# ------------------------------------------------------------
# API Keys
# ------------------------------------------------------------
try:
    from google.colab import userdata
except Exception:
    userdata = None

def get_secret(name, default=""):
    if name in os.environ and os.environ[name]:
        return os.environ[name]
    if userdata is not None:
        try:
            value = userdata.get(name)
            return value if value is not None else default
        except Exception:
            return default
    return default

API_KEYS = {
    "OPINET_CERTKEY": get_secret("OPINET_CERTKEY", ""),
    "BOK_ECOS_API_KEY": get_secret("BOK_ECOS_API_KEY", ""),
    "KOREAEXIM_AUTHKEY": get_secret("KOREAEXIM_AUTHKEY", ""),
    "GOOGLE_MAPS_API_KEY": get_secret("GOOGLE_MAPS_API_KEY", ""),
    "JUSO_CONFM_KEY": get_secret("JUSO_CONFM_KEY", ""),
    "VWORLD_KEY": get_secret("VWORLD_API_KEY", get_secret("VWORLD_KEY", "")),
    "VWORLD_DOMAIN": os.getenv("VWORLD_DOMAIN", "localhost"),
}

# ------------------------------------------------------------
# 경로
# ------------------------------------------------------------
DATA_DIR = Path(DATA_PATH)
PROCESSED_DIR = Path(PROCESSED_PATH)
ADDITIONAL_DIR = PROCESSED_DIR / "additional_data"

COLLECTION_WORK_DIR = ADDITIONAL_DIR / "_collection_work"
COLLECTION_SNAPSHOT_DIR = ADDITIONAL_DIR / "api_collection"

DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ADDITIONAL_DIR.mkdir(parents=True, exist_ok=True)
COLLECTION_WORK_DIR.mkdir(parents=True, exist_ok=True)
COLLECTION_SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 공통 요청 설정
# ------------------------------------------------------------
REQUEST_TIMEOUT = 30
REQUEST_RETRY = 3
REQUEST_SLEEP_SEC = 0.3

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

# ------------------------------------------------------------
# 원칙
# ------------------------------------------------------------
SAME_SOURCE_SAME_SHAPE_RULE = """
1. 현재 코드가 읽는 파일명, 컬럼명, 날짜 형식, 단위를 유지한다.
2. 현재 코드가 사용한 출처를 유지한다.
3. 같은 출처의 API가 있으면 API를 우선한다.
4. API로 같은 형태를 만들 수 없으면 같은 출처의 CSV/Excel 다운로드 또는 HTML 크롤링을 사용한다.
5. 둘 다 불가능하면 자동수집 불가로 두고 관리형 파일로 유지한다.
"""

print("COLLECT_START_DATE =", COLLECT_START_DATE)
print("COLLECT_END_DATE   =", COLLECT_END_DATE)
print("RUN_COLLECTION     =", RUN_COLLECTION)
print("DATA_DIR           =", DATA_DIR)
print("PROCESSED_DIR      =", PROCESSED_DIR)
print("ADDITIONAL_DIR     =", ADDITIONAL_DIR)

### 환율

In [ ]:
# ============================================================
# B-1. 환율 fx_usdkrw.csv 수집 / 전처리 / 저장
# ============================================================

# 현재 사용 파일:
#   DATA_PATH/fx_usdkrw.csv
#
# 현재 코드가 기대하는 원본 형태:
#   변환, 원자료
#
# 전처리 방식:
#   변환 -> date
#   원자료 -> 쉼표/따옴표 제거 -> numeric -> usdkrw
#
# 출처:
#   ECOS(한국은행 경제통계시스템)
#   3.1.1.1. 주요국 통화의 대원화환율
#   원/미국달러(매매기준율)
#
# ECOS 코드:
#   STAT_CODE = 731Y001
#   ITEM_CODE = 0000001
#   CYCLE = D

FX_COLLECTION_DIR = Path(DATA_COLLECTION_PATH) / "fx_usdkrw"
FX_RAW_DIR = FX_COLLECTION_DIR / "raw"
FX_FINAL_DIR = FX_COLLECTION_DIR / "final"

FX_COLLECTION_DIR.mkdir(parents=True, exist_ok=True)
FX_RAW_DIR.mkdir(parents=True, exist_ok=True)
FX_FINAL_DIR.mkdir(parents=True, exist_ok=True)

FX_FINAL_PATH = Path(DATA_PATH) / "fx_usdkrw.csv"

ECOS_FX_STAT_CODE = "731Y001"
ECOS_FX_ITEM_CODE = "0000001"
ECOS_FX_CYCLE = "D"

print("FX_COLLECTION_DIR =", FX_COLLECTION_DIR)
print("FX_FINAL_PATH     =", FX_FINAL_PATH)

In [ ]:
def _yyyymmdd(x) -> str:
    return pd.Timestamp(x).strftime("%Y%m%d")


def _request_json_with_retry(url: str, timeout: int = REQUEST_TIMEOUT, retry: int = REQUEST_RETRY):
    last_error = None

    for attempt in range(1, retry + 1):
        try:
            response = requests.get(url, headers=HEADERS, timeout=timeout)
            response.raise_for_status()
            return response.json()
        except Exception as e:
            last_error = e
            if attempt < retry:
                time.sleep(REQUEST_SLEEP_SEC * attempt)

    raise RuntimeError(f"API 요청 실패: {url}\n마지막 오류: {last_error}")


def collect_fx_usdkrw_ecos(
    start_date: str,
    end_date: str | None = None,
    api_key: str | None = None,
    page_size: int = 1000,
) -> pd.DataFrame:
    """
    ECOS 원/미국달러(매매기준율)를 수집한다.

    입력:
      start_date: YYYY-MM-DD 또는 YYYYMMDD
      end_date: YYYY-MM-DD 또는 YYYYMMDD. None이면 start_date 단일일 조회
      api_key: ECOS API key. None이면 API_KEYS 또는 환경변수에서 찾음

    출력:
      ECOS 원천 응답 row DataFrame
    """

    end_date = end_date or start_date
    start_ymd = _yyyymmdd(start_date)
    end_ymd = _yyyymmdd(end_date)

    key = (
        api_key
        or API_KEYS.get("BOK_ECOS_API_KEY")
        or os.getenv("BOK_ECOS_API_KEY", "")
    )

    if not key:
        raise ValueError("BOK_ECOS_API_KEY가 없습니다. ECOS API 키를 입력하세요.")

    first_url = (
        f"https://ecos.bok.or.kr/api/StatisticSearch/{key}/json/kr/"
        f"1/{page_size}/{ECOS_FX_STAT_CODE}/{ECOS_FX_CYCLE}/"
        f"{start_ymd}/{end_ymd}/{ECOS_FX_ITEM_CODE}"
    )

    first = _request_json_with_retry(first_url)

    if "RESULT" in first and "StatisticSearch" not in first:
        raise RuntimeError(f"ECOS 오류 응답: {first}")

    total_count = int(first.get("StatisticSearch", {}).get("list_total_count", 0))

    if total_count == 0:
        return pd.DataFrame()

    rows = []

    for start_idx in range(1, total_count + 1, page_size):
        end_idx = min(start_idx + page_size - 1, total_count)

        url = (
            f"https://ecos.bok.or.kr/api/StatisticSearch/{key}/json/kr/"
            f"{start_idx}/{end_idx}/{ECOS_FX_STAT_CODE}/{ECOS_FX_CYCLE}/"
            f"{start_ymd}/{end_ymd}/{ECOS_FX_ITEM_CODE}"
        )

        data = _request_json_with_retry(url)

        if "RESULT" in data and "StatisticSearch" not in data:
            raise RuntimeError(f"ECOS 오류 응답: {data}")

        page_rows = data.get("StatisticSearch", {}).get("row", [])
        rows.extend(page_rows)

        time.sleep(REQUEST_SLEEP_SEC)

    raw = pd.DataFrame(rows)

    if raw.empty:
        return raw

    raw = (
        raw.drop_duplicates(subset=["TIME", "ITEM_CODE1"], keep="last")
           .sort_values("TIME")
           .reset_index(drop=True)
    )

    return raw

In [ ]:
def preprocess_fx_usdkrw_for_current_code(raw_fx: pd.DataFrame) -> pd.DataFrame:
    """
    ECOS 원천 응답을 현재 코드가 읽는 fx_usdkrw.csv 형태로 변환한다.

    현재 코드 기대 형태:
      변환, 원자료

    변환 예:
      TIME=20200110 -> 변환=2020/01/10
      DATA_VALUE=1160 -> 원자료=1160.0
    """

    if raw_fx is None or raw_fx.empty:
        raise ValueError("환율 원천 데이터가 비어 있습니다.")

    required = ["TIME", "DATA_VALUE"]
    missing = [c for c in required if c not in raw_fx.columns]

    if missing:
        raise ValueError(f"ECOS 환율 응답 필수 컬럼 누락: {missing}")

    out = pd.DataFrame({
        "변환": pd.to_datetime(raw_fx["TIME"], format="%Y%m%d", errors="coerce").dt.strftime("%Y/%m/%d"),
        "원자료": (
            raw_fx["DATA_VALUE"]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace('"', "", regex=False)
            .str.strip()
        ),
    })

    out["원자료"] = pd.to_numeric(out["원자료"], errors="coerce")

    out = (
        out.dropna(subset=["변환", "원자료"])
           .drop_duplicates(subset=["변환"], keep="last")
           .sort_values("변환")
           .reset_index(drop=True)
    )

    return out

In [ ]:
def save_fx_usdkrw_outputs(
    raw_fx: pd.DataFrame,
    final_fx: pd.DataFrame,
    start_date: str,
    end_date: str | None = None,
    save_to_data_path: bool = True,
) -> dict:
    """
    환율 수집 결과를 저장한다.

    저장 위치:
      1. DATA_COLLECTION_PATH/fx_usdkrw/raw/
      2. DATA_COLLECTION_PATH/fx_usdkrw/final/
      3. DATA_PATH/fx_usdkrw.csv
    """

    end_date = end_date or start_date
    stamp = f"{_yyyymmdd(start_date)}_{_yyyymmdd(end_date)}"

    raw_path = FX_RAW_DIR / f"ecos_fx_usdkrw_raw_{stamp}.csv"
    final_snapshot_path = FX_FINAL_DIR / f"fx_usdkrw_{stamp}.csv"

    raw_fx.to_csv(raw_path, index=False, encoding="utf-8-sig")
    final_fx.to_csv(final_snapshot_path, index=False, encoding="utf-8-sig")

    saved = {
        "raw_path": raw_path,
        "final_snapshot_path": final_snapshot_path,
    }

    if save_to_data_path:
        final_fx.to_csv(FX_FINAL_PATH, index=False, encoding="utf-8-sig")
        saved["final_data_path"] = FX_FINAL_PATH

    print("[환율 저장 완료]")
    for k, v in saved.items():
        print(f"- {k}: {v}")

    print(f"- rows: {len(final_fx):,}")
    print(f"- date_min: {final_fx['변환'].min()}")
    print(f"- date_max: {final_fx['변환'].max()}")

    return saved


def validate_fx_usdkrw_current_shape(path: str | Path = FX_FINAL_PATH) -> pd.DataFrame:
    """
    현재 01 전처리 코드의 read_fx 방식과 동일하게 읽히는지 확인한다.
    """

    df = pd.read_csv(path, encoding="utf-8-sig")

    required = ["변환", "원자료"]
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(f"fx_usdkrw.csv 필수 컬럼 누락: {missing}")

    check = df.copy()
    check["date"] = pd.to_datetime(check["변환"], errors="coerce")
    check["usdkrw"] = (
        check["원자료"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace('"', "", regex=False)
        .str.strip()
    )
    check["usdkrw"] = pd.to_numeric(check["usdkrw"], errors="coerce")

    if check["date"].isna().any():
        raise ValueError("날짜 파싱 실패 행이 있습니다.")

    if check["usdkrw"].isna().any():
        raise ValueError("환율 숫자 변환 실패 행이 있습니다.")

    report = pd.DataFrame([{
        "path": str(path),
        "rows": len(check),
        "date_min": check["date"].min(),
        "date_max": check["date"].max(),
        "usdkrw_min": check["usdkrw"].min(),
        "usdkrw_max": check["usdkrw"].max(),
    }])

    display(report)
    return report

In [ ]:
def collect_preprocess_save_fx_usdkrw(
    start_date: str,
    end_date: str | None = None,
    api_key: str | None = None,
    save_to_data_path: bool = True,
) -> dict:
    """
    환율 수집 -> 현재 파일 형태 전처리 -> 저장 -> 검증까지 한 번에 수행한다.

    단일일:
      collect_preprocess_save_fx_usdkrw("2026-06-09")

    기간:
      collect_preprocess_save_fx_usdkrw("2008-01-01", "2026-06-09")
    """

    raw_fx = collect_fx_usdkrw_ecos(
        start_date=start_date,
        end_date=end_date,
        api_key=api_key,
    )

    final_fx = preprocess_fx_usdkrw_for_current_code(raw_fx)

    saved = save_fx_usdkrw_outputs(
        raw_fx=raw_fx,
        final_fx=final_fx,
        start_date=start_date,
        end_date=end_date,
        save_to_data_path=save_to_data_path,
    )

    validate_fx_usdkrw_current_shape(
        saved["final_data_path"] if save_to_data_path else saved["final_snapshot_path"]
    )

    return {
        "raw": raw_fx,
        "final": final_fx,
        "saved": saved,
    }

In [ ]:
# ============================================================
# B-1 실행
# ============================================================

if RUN_COLLECTION and RUN_FX:
    FX_RESULT = collect_preprocess_save_fx_usdkrw(
        start_date=COLLECT_START_DATE,
        end_date=COLLECT_END_DATE,
        api_key=API_KEYS.get("BOK_ECOS_API_KEY"),
        save_to_data_path=True,
    )
else:
    print("RUN_COLLECTION 또는 RUN_FX가 False라서 환율 수집을 실행하지 않았습니다.")

### 원유가

In [ ]:
# ============================================================
# B-2-0. 원유가 crude.csv 설정
# ============================================================

import io
import re

CRUDE_COLLECTION_DIR = Path(DATA_COLLECTION_PATH) / "crude"
CRUDE_RAW_DIR = CRUDE_COLLECTION_DIR / "raw"
CRUDE_FINAL_DIR = CRUDE_COLLECTION_DIR / "final"

CRUDE_COLLECTION_DIR.mkdir(parents=True, exist_ok=True)
CRUDE_RAW_DIR.mkdir(parents=True, exist_ok=True)
CRUDE_FINAL_DIR.mkdir(parents=True, exist_ok=True)

CRUDE_FINAL_PATH = Path(DATA_PATH) / "crude.csv"

OPINET_CRUDE_PAGE_URL = "https://www.opinet.co.kr/glopcoilSelect.do"
OPINET_CRUDE_CSV_URL = "https://www.opinet.co.kr/glopcoil_csv.do"

print("CRUDE_COLLECTION_DIR =", CRUDE_COLLECTION_DIR)
print("CRUDE_FINAL_PATH     =", CRUDE_FINAL_PATH)

In [ ]:
def _opinet_date_parts(x):
    ts = pd.Timestamp(x)
    return ts.strftime("%Y"), ts.strftime("%m"), ts.strftime("%d"), ts.strftime("%Y%m%d")


def _read_opinet_csv_bytes(raw_bytes: bytes) -> pd.DataFrame:
    last_error = None
    for enc in ["euc-kr", "cp949", "utf-8-sig", "utf-8"]:
        try:
            return pd.read_csv(io.BytesIO(raw_bytes), encoding=enc)
        except Exception as e:
            last_error = e
    raise RuntimeError(f"오피넷 CSV 디코딩 실패: {last_error}")


def _parse_opinet_crude_date(x):
    s = str(x).replace("\xa0", "").replace(" ", "").strip()
    m = re.match(r"(\d{2,4})년(\d{1,2})월(\d{1,2})일", s)
    if not m:
        raise ValueError(f"원유 날짜 파싱 실패: {x}")

    y = int(m.group(1))
    if y < 100:
        y += 2000 if y < 50 else 1900

    return pd.Timestamp(y, int(m.group(2)), int(m.group(3)))


def collect_crude_opinet_csv(
    start_date: str,
    end_date: str | None = None,
    unit: str = "usd",
) -> tuple[pd.DataFrame, bytes]:
    """
    오피넷 원유 CSV를 수집한다.

    unit:
      "usd" -> $/Bbl, 현재 코드 기준. 기본값.
      "krw" -> 원 단위. 현재 모델에는 사용하지 않음.
    """

    end_date = end_date or start_date

    sta_y, sta_m, sta_d, stddate = _opinet_date_parts(start_date)
    end_y, end_m, end_d, enddate = _opinet_date_parts(end_date)

    if unit not in ["usd", "krw"]:
        raise ValueError("unit은 'usd' 또는 'krw'만 가능합니다.")

    sel_div = "div_dar" if unit == "usd" else "div_won"

    payload = [
        ("TERM", "D"),
        ("STA_Y", sta_y),
        ("STA_M", sta_m),
        ("STA_D", sta_d),
        ("END_Y", end_y),
        ("END_M", end_m),
        ("END_D", end_d),
        ("OILSRTCD", "001"),  # Dubai
        ("OILSRTCD", "002"),  # Brent
        ("OILSRTCD", "003"),  # WTI
        ("OILSRTCD1", "001"),
        ("OILSRTCD2", "002"),
        ("OILSRTCD3", "003"),
        ("STDDATE", stddate),
        ("ENDDATE", enddate),
        ("SEL_DIV", sel_div),
    ]

    session = requests.Session()
    session.headers.update(HEADERS)

    # 첫 페이지 호출로 세션/기본 상태 확보
    session.get(OPINET_CRUDE_PAGE_URL, timeout=REQUEST_TIMEOUT)

    last_error = None
    for attempt in range(1, REQUEST_RETRY + 1):
        try:
            response = session.post(
                OPINET_CRUDE_CSV_URL,
                data=payload,
                timeout=REQUEST_TIMEOUT,
            )
            response.raise_for_status()
            raw_bytes = response.content

            raw_df = _read_opinet_csv_bytes(raw_bytes)

            if "기간" not in raw_df.columns:
                raise RuntimeError(f"오피넷 원유 CSV 응답에 '기간' 컬럼이 없습니다: {raw_df.columns.tolist()}")

            return raw_df, raw_bytes

        except Exception as e:
            last_error = e
            if attempt < REQUEST_RETRY:
                time.sleep(REQUEST_SLEEP_SEC * attempt)

    raise RuntimeError(f"오피넷 원유 CSV 수집 실패: {last_error}")


def preprocess_crude_for_current_code(raw_crude: pd.DataFrame) -> pd.DataFrame:
    """
    오피넷 원유 CSV를 현재 코드가 읽는 crude.csv 형태로 정리한다.

    최종 형태:
      기간, Dubai, Brent, WTI
    """

    if raw_crude is None or raw_crude.empty:
        raise ValueError("원유 원천 데이터가 비어 있습니다.")

    df = raw_crude.copy()
    df.columns = [str(c).strip() for c in df.columns]

    required = ["기간", "Dubai", "Brent", "WTI"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"원유 필수 컬럼 누락: {missing}")

    out = df[required].copy()
    out["기간"] = out["기간"].astype(str).str.replace("\xa0", "", regex=False).str.strip()

    for c in ["Dubai", "Brent", "WTI"]:
        out[c] = pd.to_numeric(
            out[c].astype(str).str.replace(",", "", regex=False).str.strip(),
            errors="coerce",
        )

    out["_date"] = out["기간"].apply(_parse_opinet_crude_date)
    out = (
        out.dropna(subset=["Dubai", "Brent", "WTI"], how="all")
           .drop_duplicates(subset=["기간"], keep="last")
           .sort_values("_date")
           .drop(columns=["_date"])
           .reset_index(drop=True)
    )

    return out

In [ ]:
def save_crude_outputs(
    raw_bytes: bytes,
    final_crude: pd.DataFrame,
    start_date: str,
    end_date: str | None = None,
    save_to_data_path: bool = True,
) -> dict:
    end_date = end_date or start_date
    stamp = f"{_yyyymmdd(start_date)}_{_yyyymmdd(end_date)}"

    raw_path = CRUDE_RAW_DIR / f"opinet_crude_raw_{stamp}.csv"
    final_snapshot_path = CRUDE_FINAL_DIR / f"crude_{stamp}.csv"

    raw_path.write_bytes(raw_bytes)
    final_crude.to_csv(final_snapshot_path, index=False, encoding="utf-8-sig")

    saved = {
        "raw_path": raw_path,
        "final_snapshot_path": final_snapshot_path,
    }

    if save_to_data_path:
        final_crude.to_csv(CRUDE_FINAL_PATH, index=False, encoding="utf-8-sig")
        saved["final_data_path"] = CRUDE_FINAL_PATH

    print("[원유가 저장 완료]")
    for k, v in saved.items():
        print(f"- {k}: {v}")

    print(f"- rows: {len(final_crude):,}")
    print(f"- date_min: {final_crude['기간'].iloc[0] if len(final_crude) else None}")
    print(f"- date_max: {final_crude['기간'].iloc[-1] if len(final_crude) else None}")

    return saved


def validate_crude_current_shape(path: str | Path = CRUDE_FINAL_PATH) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")

    required = ["기간", "Dubai", "Brent", "WTI"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"crude.csv 필수 컬럼 누락: {missing}")

    check = df.copy()
    check["date"] = check["기간"].apply(_parse_opinet_crude_date)

    for c in ["Dubai", "Brent", "WTI"]:
        check[c] = pd.to_numeric(check[c], errors="coerce")

    duplicate_dates = int(check["date"].duplicated().sum())
    if duplicate_dates > 0:
        raise ValueError(f"중복 날짜가 있습니다: {duplicate_dates}")

    report = pd.DataFrame([{
        "path": str(path),
        "rows": len(check),
        "date_min": check["date"].min(),
        "date_max": check["date"].max(),
        "dubai_min": check["Dubai"].min(),
        "dubai_max": check["Dubai"].max(),
        "brent_min": check["Brent"].min(),
        "brent_max": check["Brent"].max(),
        "wti_min": check["WTI"].min(),
        "wti_max": check["WTI"].max(),
        "duplicate_dates": duplicate_dates,
    }])

    display(report)
    return report


def collect_preprocess_save_crude(
    start_date: str,
    end_date: str | None = None,
    save_to_data_path: bool = True,
) -> dict:
    """
    원유가 수집 -> 현재 파일 형태 전처리 -> 저장 -> 검증.

    단일일:
      collect_preprocess_save_crude("2026-06-09")

    기간:
      collect_preprocess_save_crude("2008-01-01", "2026-06-10")
    """

    raw_crude, raw_bytes = collect_crude_opinet_csv(
        start_date=start_date,
        end_date=end_date,
        unit="usd",
    )

    final_crude = preprocess_crude_for_current_code(raw_crude)

    saved = save_crude_outputs(
        raw_bytes=raw_bytes,
        final_crude=final_crude,
        start_date=start_date,
        end_date=end_date,
        save_to_data_path=save_to_data_path,
    )

    validate_crude_current_shape(
        saved["final_data_path"] if save_to_data_path else saved["final_snapshot_path"]
    )

    return {
        "raw": raw_crude,
        "final": final_crude,
        "saved": saved,
    }

In [ ]:
# ============================================================
# B-2 실행
# ============================================================

if RUN_COLLECTION and RUN_CRUDE:
    CRUDE_RESULT = collect_preprocess_save_crude(
        start_date=COLLECT_START_DATE,
        end_date=COLLECT_END_DATE,
        save_to_data_path=True,
    )
else:
    print("RUN_COLLECTION 또는 RUN_CRUDE가 False라서 원유가 수집을 실행하지 않았습니다.")

### 국제 석유제품

In [ ]:
# ============================================================
# B-3-0. 국제 석유제품 설정
# ============================================================

import io
import re

INTL_PRODUCT_COLLECTION_DIR = Path(DATA_COLLECTION_PATH) / "intl_products"
INTL_PRODUCT_RAW_DIR = INTL_PRODUCT_COLLECTION_DIR / "raw"
INTL_PRODUCT_FINAL_DIR = INTL_PRODUCT_COLLECTION_DIR / "final"

INTL_PRODUCT_COLLECTION_DIR.mkdir(parents=True, exist_ok=True)
INTL_PRODUCT_RAW_DIR.mkdir(parents=True, exist_ok=True)
INTL_PRODUCT_FINAL_DIR.mkdir(parents=True, exist_ok=True)

INTL_PRODUCTS_CSV_FINAL_PATH = Path(DATA_PATH) / "intl_products.csv"
INTL_DIESEL001_FINAL_PATH = Path(DATA_PATH) / "intl_product_diesel(0.001).csv"

OPINET_INTL_PRODUCT_PAGE_URL = "https://www.opinet.co.kr/glopopdSelect.do"
OPINET_INTL_PRODUCT_CSV_URL = "https://www.opinet.co.kr/glopopd_csv.do"

OPINET_INTL_PRODUCT_CODES = [
    ("B001", "휘발유(95RON)"),
    ("B007", "휘발유(92RON)"),
    ("C001", "등유"),
    ("D009", "경유(0.001%)"),
    ("D008", "경유(0.05%)"),
    ("E001", "고유황중유(180cst/3.5%)"),
    ("F001", "나프타"),
]

In [ ]:
# ============================================================
# B-3-1. 국제 석유제품 수집 함수
# ============================================================

def _intl_yyyymmdd(x) -> str:
    return pd.Timestamp(x).strftime("%Y%m%d")


def _intl_date_parts(x):
    ts = pd.Timestamp(x)
    return ts.strftime("%Y"), ts.strftime("%m"), ts.strftime("%d"), ts.strftime("%Y%m%d")


def _read_opinet_intl_csv_bytes(raw_bytes: bytes) -> pd.DataFrame:
    last_error = None
    for enc in ["euc-kr", "cp949", "utf-8-sig", "utf-8"]:
        try:
            return pd.read_csv(io.BytesIO(raw_bytes), encoding=enc)
        except Exception as e:
            last_error = e
    raise RuntimeError(f"오피넷 국제제품 CSV 디코딩 실패: {last_error}")


def collect_intl_products_opinet_csv(start_date, end_date=None, unit="usd", include_holiday=True):
    end_date = end_date or start_date

    sta_y, sta_m, sta_d, stddate = _intl_date_parts(start_date)
    end_y, end_m, end_d, enddate = _intl_date_parts(end_date)

    sel_div = "div_dar" if unit == "usd" else "div_won"
    holiday_yn = "Y" if include_holiday else "N"

    payload = [
        ("TERM", "D"),
        ("HOLIDAY_YN", holiday_yn),
        ("STA_Y", sta_y),
        ("STA_M", sta_m),
        ("STA_D", sta_d),
        ("END_Y", end_y),
        ("END_M", end_m),
        ("END_D", end_d),
    ]

    for code, _ in OPINET_INTL_PRODUCT_CODES:
        payload.append(("OILSRTCD", code))

    for idx, (code, _) in enumerate(OPINET_INTL_PRODUCT_CODES, start=1):
        payload.append((f"OILSRTCD{idx}", code))

    payload.extend([
        ("STDDATE", stddate),
        ("ENDDATE", enddate),
        ("SEL_DIV", sel_div),
    ])

    session = requests.Session()
    session.headers.update(HEADERS)
    session.get(OPINET_INTL_PRODUCT_PAGE_URL, timeout=REQUEST_TIMEOUT)

    last_error = None
    for attempt in range(1, REQUEST_RETRY + 1):
        try:
            res = session.post(OPINET_INTL_PRODUCT_CSV_URL, data=payload, timeout=REQUEST_TIMEOUT)
            res.raise_for_status()

            raw_bytes = res.content
            raw_df = _read_opinet_intl_csv_bytes(raw_bytes)

            if "기간" not in raw_df.columns:
                raise RuntimeError(f"'기간' 컬럼 없음: {raw_df.columns.tolist()}")

            return raw_df, raw_bytes

        except Exception as e:
            last_error = e
            if attempt < REQUEST_RETRY:
                time.sleep(REQUEST_SLEEP_SEC * attempt)

    raise RuntimeError(f"오피넷 국제제품 CSV 수집 실패: {last_error}")

In [ ]:
# ============================================================
# B-3-2. 국제 석유제품 전처리 함수
# ============================================================

def _parse_opinet_intl_date(x):
    s = str(x).replace("\xa0", "").replace(" ", "").strip()
    m = re.match(r"(\d{2,4})년(\d{1,2})월(\d{1,2})일", s)

    if not m:
        raise ValueError(f"국제제품 날짜 파싱 실패: {x}")

    y = int(m.group(1))
    if y < 100:
        y += 2000 if y < 50 else 1900

    return pd.Timestamp(y, int(m.group(2)), int(m.group(3)))


def _format_short_kor_date(ts):
    ts = pd.Timestamp(ts)
    return f"{ts.year % 100:02d}년{ts.month:02d}월{ts.day:02d}일"


def preprocess_intl_products_for_current_csv(raw_products: pd.DataFrame) -> pd.DataFrame:
    df = raw_products.copy()
    df.columns = [str(c).strip() for c in df.columns]

    required = [
        "기간",
        "휘발유(95RON)",
        "휘발유(92RON)",
        "경유(0.001%)",
        "경유(0.05%)",
        "나프타",
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"국제제품 필수 컬럼 누락: {missing}")

    df["_date"] = df["기간"].apply(_parse_opinet_intl_date)

    product_cols = [c for c in df.columns if c not in ["기간", "_date"]]
    for c in product_cols:
        df[c] = pd.to_numeric(
            df[c].astype(str).str.replace(",", "", regex=False).str.strip(),
            errors="coerce",
        )

    out_cols = [
        "기간",
        "휘발유(95RON)",
        "휘발유(92RON)",
        "등유",
        "경유(0.001%)",
        "경유(0.05%)",
        "고유황중유(180cst/3.5%)",
        "나프타",
    ]
    out_cols = [c for c in out_cols if c in df.columns]

    out = (
        df.sort_values("_date")
          .drop_duplicates(subset=["_date"], keep="last")
          .assign(기간=lambda x: x["_date"].apply(_format_short_kor_date))
          [out_cols]
          .reset_index(drop=True)
    )

    return out


def build_intl_diesel001_current_shape(intl_products_csv: pd.DataFrame) -> pd.DataFrame:
    if "경유(0.001%)" not in intl_products_csv.columns:
        raise ValueError("'경유(0.001%)' 컬럼이 없습니다.")

    return intl_products_csv[["기간", "경유(0.001%)"]].copy()


def read_intl_products_csv_for_model(path: str | Path) -> pd.DataFrame:
    """
    기존 parse_product_zip() 대체용.
    01 전처리에서 intl_products.csv를 읽을 때 사용.
    """

    df = pd.read_csv(path, encoding="utf-8-sig")
    df["date"] = df["기간"].apply(_parse_opinet_intl_date)

    rename_map = {
        "휘발유(95RON)": "휘발유95RON",
        "휘발유(92RON)": "휘발유92RON",
        "경유(0.05%)": "경유0.05",
        "경유(0.001%)": "경유0.001",
    }

    df = df.rename(columns=rename_map)

    keep = [
        "date",
        "휘발유95RON",
        "휘발유92RON",
        "경유0.05",
        "경유0.001",
        "나프타",
    ]
    keep = [c for c in keep if c in df.columns]

    out = df[keep].copy()

    for c in out.columns:
        if c != "date":
            out[c] = pd.to_numeric(out[c], errors="coerce")

    out["non_nulls"] = out.drop(columns=["date"]).notna().sum(axis=1)
    out = (
        out.sort_values(["date", "non_nulls"])
           .drop_duplicates("date", keep="last")
           .drop(columns=["non_nulls"])
           .reset_index(drop=True)
    )

    return out

In [ ]:
# ============================================================
# B-3-3. 저장 / 검증 함수
# ============================================================

def save_intl_products_outputs(raw_bytes, final_products, final_diesel001, start_date, end_date=None, save_to_data_path=True):
    end_date = end_date or start_date
    stamp = f"{_intl_yyyymmdd(start_date)}_{_intl_yyyymmdd(end_date)}"

    raw_path = INTL_PRODUCT_RAW_DIR / f"opinet_intl_products_raw_{stamp}.csv"
    final_snapshot_path = INTL_PRODUCT_FINAL_DIR / f"intl_products_{stamp}.csv"
    diesel001_snapshot_path = INTL_PRODUCT_FINAL_DIR / f"intl_product_diesel(0.001)_{stamp}.csv"

    raw_path.write_bytes(raw_bytes)
    final_products.to_csv(final_snapshot_path, index=False, encoding="utf-8-sig")
    final_diesel001.to_csv(diesel001_snapshot_path, index=False, encoding="utf-8-sig")

    saved = {
        "raw_path": raw_path,
        "final_snapshot_path": final_snapshot_path,
        "diesel001_snapshot_path": diesel001_snapshot_path,
    }

    if save_to_data_path:
        final_products.to_csv(INTL_PRODUCTS_CSV_FINAL_PATH, index=False, encoding="utf-8-sig")
        final_diesel001.to_csv(INTL_DIESEL001_FINAL_PATH, index=False, encoding="utf-8-sig")
        saved["final_products_data_path"] = INTL_PRODUCTS_CSV_FINAL_PATH
        saved["final_diesel001_data_path"] = INTL_DIESEL001_FINAL_PATH

    print("[국제 석유제품 저장 완료]")
    for k, v in saved.items():
        print(f"- {k}: {v}")

    print(f"- rows: {len(final_products):,}")
    print(f"- date_min: {final_products['기간'].iloc[0]}")
    print(f"- date_max: {final_products['기간'].iloc[-1]}")

    return saved


def validate_intl_products_current_shape(path=INTL_PRODUCTS_CSV_FINAL_PATH):
    df = pd.read_csv(path, encoding="utf-8-sig")

    required = [
        "기간",
        "휘발유(95RON)",
        "휘발유(92RON)",
        "경유(0.001%)",
        "경유(0.05%)",
        "나프타",
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"intl_products.csv 필수 컬럼 누락: {missing}")

    dates = df["기간"].apply(_parse_opinet_intl_date)
    duplicate_dates = int(dates.duplicated().sum())

    if duplicate_dates > 0:
        raise ValueError(f"중복 날짜가 있습니다: {duplicate_dates}")

    model_df = read_intl_products_csv_for_model(path)

    report = pd.DataFrame([{
        "path": str(path),
        "rows": len(df),
        "date_min": dates.min(),
        "date_max": dates.max(),
        "duplicate_dates": duplicate_dates,
        "model_columns": ", ".join(model_df.columns),
        "model_rows": len(model_df),
    }])

    display(report)
    return report

In [ ]:
# ============================================================
# B-3-4. 통합 실행 함수
# ============================================================

def collect_preprocess_save_intl_products(start_date, end_date=None, save_to_data_path=True):
    raw_products, raw_bytes = collect_intl_products_opinet_csv(
        start_date=start_date,
        end_date=end_date,
        unit="usd",
        include_holiday=True,
    )

    final_products = preprocess_intl_products_for_current_csv(raw_products)
    final_diesel001 = build_intl_diesel001_current_shape(final_products)

    saved = save_intl_products_outputs(
        raw_bytes=raw_bytes,
        final_products=final_products,
        final_diesel001=final_diesel001,
        start_date=start_date,
        end_date=end_date,
        save_to_data_path=save_to_data_path,
    )

    validate_intl_products_current_shape(
        saved["final_products_data_path"] if save_to_data_path else saved["final_snapshot_path"]
    )

    return {
        "raw": raw_products,
        "final": final_products,
        "diesel001": final_diesel001,
        "saved": saved,
    }

In [ ]:
# ============================================================
# B-3 실행
# ============================================================

if RUN_COLLECTION and RUN_INTL_PRODUCTS:
    INTL_PRODUCTS_RESULT = collect_preprocess_save_intl_products(
        start_date=COLLECT_START_DATE,
        end_date=COLLECT_END_DATE,
        save_to_data_path=True,
    )
else:
    print("RUN_COLLECTION 또는 RUN_INTL_PRODUCTS가 False라서 국제 석유제품 수집을 실행하지 않았습니다.")

### 전국 평균 소매가

In [ ]:
# ============================================================
# B-4-0. 전국 평균 소매가 retail_avg.csv 설정
# ============================================================

import io
import re

RETAIL_AVG_COLLECTION_DIR = Path(DATA_COLLECTION_PATH) / "retail_avg"
RETAIL_AVG_RAW_DIR = RETAIL_AVG_COLLECTION_DIR / "raw"
RETAIL_AVG_FINAL_DIR = RETAIL_AVG_COLLECTION_DIR / "final"

RETAIL_AVG_COLLECTION_DIR.mkdir(parents=True, exist_ok=True)
RETAIL_AVG_RAW_DIR.mkdir(parents=True, exist_ok=True)
RETAIL_AVG_FINAL_DIR.mkdir(parents=True, exist_ok=True)

RETAIL_AVG_FINAL_PATH = Path(DATA_PATH) / "retail_avg.csv"

OPINET_RETAIL_AVG_PAGE_URL = "https://www.opinet.co.kr/user/dopospdrg/dopOsPdrgSelect.do"
OPINET_RETAIL_AVG_CSV_URL = "https://www.opinet.co.kr/user/dopospdrg/dopOsPdrgCsv.do"

print("RETAIL_AVG_COLLECTION_DIR =", RETAIL_AVG_COLLECTION_DIR)
print("RETAIL_AVG_FINAL_PATH     =", RETAIL_AVG_FINAL_PATH)

In [ ]:
# ============================================================
# B-4-1. 전국 평균 소매가 수집 함수
# ============================================================

def _retail_yyyymmdd(x) -> str:
    return pd.Timestamp(x).strftime("%Y%m%d")


def _retail_date_parts(x):
    ts = pd.Timestamp(x)
    return ts.strftime("%Y"), ts.strftime("%m"), ts.strftime("%d"), ts.strftime("%Y%m%d")


def _read_opinet_retail_csv_bytes(raw_bytes: bytes) -> pd.DataFrame:
    last_error = None
    for enc in ["euc-kr", "cp949", "utf-8-sig", "utf-8"]:
        try:
            return pd.read_csv(io.BytesIO(raw_bytes), encoding=enc)
        except Exception as e:
            last_error = e
    raise RuntimeError(f"오피넷 평균판매가격 CSV 디코딩 실패: {last_error}")


def collect_retail_avg_opinet_csv(start_date, end_date=None):
    """
    오피넷 제품별 평균판매가격 CSV를 수집한다.

    수집 제품:
      - 보통휘발유
      - 자동차용경유

    단일일:
      collect_retail_avg_opinet_csv("2026-06-09")

    기간:
      collect_retail_avg_opinet_csv("2008-01-01", "2026-06-10")
    """

    end_date = end_date or start_date

    sta_y, sta_m, sta_d, stddate = _retail_date_parts(start_date)
    end_y, end_m, end_d, enddate = _retail_date_parts(end_date)

    payload = [
        ("all_chk_cnt", "5"),
        ("INIF_FLAG", "N"),
        ("chk_cnt", "2"),
        ("h_maxYY", pd.Timestamp(end_date).strftime("%Y")),
        ("h_maxQQ", ""),
        ("h_maxMM", pd.Timestamp(end_date).strftime("%Y%m")),
        ("h_maxDD", enddate),
        ("h_maxWW", ""),
        ("sta_dt", ""),
        ("end_dt", ""),
        ("TERM", "D"),
        ("STA_Y", sta_y),
        ("STA_M", sta_m),
        ("STA_D", sta_d),
        ("END_Y", end_y),
        ("END_M", end_m),
        ("END_D", end_d),
        ("OIL_CD_B027", "Y"),  # 보통휘발유
        ("OIL_CD_D047", "Y"),  # 자동차용경유
    ]

    session = requests.Session()
    session.headers.update(HEADERS)

    session.get(OPINET_RETAIL_AVG_PAGE_URL, timeout=REQUEST_TIMEOUT)

    last_error = None
    for attempt in range(1, REQUEST_RETRY + 1):
        try:
            response = session.post(
                OPINET_RETAIL_AVG_CSV_URL,
                data=payload,
                timeout=REQUEST_TIMEOUT,
            )
            response.raise_for_status()

            raw_bytes = response.content
            raw_df = _read_opinet_retail_csv_bytes(raw_bytes)

            if "구분" not in raw_df.columns:
                raise RuntimeError(f"오피넷 평균판매가격 CSV에 '구분' 컬럼이 없습니다: {raw_df.columns.tolist()}")

            return raw_df, raw_bytes

        except Exception as e:
            last_error = e
            if attempt < REQUEST_RETRY:
                time.sleep(REQUEST_SLEEP_SEC * attempt)

    raise RuntimeError(f"오피넷 평균판매가격 CSV 수집 실패: {last_error}")

In [ ]:
# ============================================================
# B-4-2. 전국 평균 소매가 전처리 함수
# ============================================================

def _parse_opinet_retail_date(x):
    s = str(x).replace("\xa0", "").replace(" ", "").strip()

    m = re.match(r"(\d{2,4})년(\d{1,2})월(\d{1,2})일", s)
    if not m:
        raise ValueError(f"평균판매가격 날짜 파싱 실패: {x}")

    y = int(m.group(1))
    if y < 100:
        y += 2000 if y < 50 else 1900

    return pd.Timestamp(y, int(m.group(2)), int(m.group(3)))


def _format_full_kor_date(ts):
    ts = pd.Timestamp(ts)
    return f"{ts.year:04d}년{ts.month:02d}월{ts.day:02d}일"


def preprocess_retail_avg_for_current_code(raw_retail: pd.DataFrame) -> pd.DataFrame:
    """
    오피넷 제품별 평균판매가격 CSV를 현재 코드가 읽는 retail_avg.csv 형태로 정리한다.

    최종 컬럼:
      구분, 보통휘발유, 자동차용경유
    """

    if raw_retail is None or raw_retail.empty:
        raise ValueError("전국 평균 소매가 원천 데이터가 비어 있습니다.")

    df = raw_retail.copy()
    df.columns = [str(c).strip() for c in df.columns]

    required = ["구분", "보통휘발유", "자동차용경유"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"전국 평균 소매가 필수 컬럼 누락: {missing}")

    out = df[required].copy()
    out["_date"] = out["구분"].apply(_parse_opinet_retail_date)

    for c in ["보통휘발유", "자동차용경유"]:
        out[c] = pd.to_numeric(
            out[c].astype(str).str.replace(",", "", regex=False).str.strip(),
            errors="coerce",
        )

    out = (
        out.dropna(subset=["보통휘발유", "자동차용경유"], how="all")
           .sort_values("_date")
           .drop_duplicates(subset=["_date"], keep="last")
           .assign(구분=lambda x: x["_date"].apply(_format_full_kor_date))
           [["구분", "보통휘발유", "자동차용경유"]]
           .reset_index(drop=True)
    )

    return out

In [ ]:
# ============================================================
# B-4-3. 저장 / 검증 함수
# ============================================================

def save_retail_avg_outputs(raw_bytes, final_retail, start_date, end_date=None, save_to_data_path=True):
    end_date = end_date or start_date
    stamp = f"{_retail_yyyymmdd(start_date)}_{_retail_yyyymmdd(end_date)}"

    raw_path = RETAIL_AVG_RAW_DIR / f"opinet_retail_avg_raw_{stamp}.csv"
    final_snapshot_path = RETAIL_AVG_FINAL_DIR / f"retail_avg_{stamp}.csv"

    raw_path.write_bytes(raw_bytes)
    final_retail.to_csv(final_snapshot_path, index=False, encoding="utf-8-sig")

    saved = {
        "raw_path": raw_path,
        "final_snapshot_path": final_snapshot_path,
    }

    if save_to_data_path:
        final_retail.to_csv(RETAIL_AVG_FINAL_PATH, index=False, encoding="utf-8-sig")
        saved["final_data_path"] = RETAIL_AVG_FINAL_PATH

    print("[전국 평균 소매가 저장 완료]")
    for k, v in saved.items():
        print(f"- {k}: {v}")

    print(f"- rows: {len(final_retail):,}")
    print(f"- date_min: {final_retail['구분'].iloc[0] if len(final_retail) else None}")
    print(f"- date_max: {final_retail['구분'].iloc[-1] if len(final_retail) else None}")

    return saved


def validate_retail_avg_current_shape(path=RETAIL_AVG_FINAL_PATH):
    df = pd.read_csv(path, encoding="utf-8-sig")

    required = ["구분", "보통휘발유", "자동차용경유"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"retail_avg.csv 필수 컬럼 누락: {missing}")

    check = df.copy()
    check["date"] = check["구분"].apply(_parse_opinet_retail_date)

    for c in ["보통휘발유", "자동차용경유"]:
        check[c] = pd.to_numeric(check[c], errors="coerce")

    duplicate_dates = int(check["date"].duplicated().sum())
    if duplicate_dates > 0:
        raise ValueError(f"중복 날짜가 있습니다: {duplicate_dates}")

    null_rows = int(check[["보통휘발유", "자동차용경유"]].isna().all(axis=1).sum())
    if null_rows > 0:
        raise ValueError(f"휘발유/경유가 모두 비어 있는 행이 있습니다: {null_rows}")

    report = pd.DataFrame([{
        "path": str(path),
        "rows": len(check),
        "date_min": check["date"].min(),
        "date_max": check["date"].max(),
        "gasoline_min": check["보통휘발유"].min(),
        "gasoline_max": check["보통휘발유"].max(),
        "diesel_min": check["자동차용경유"].min(),
        "diesel_max": check["자동차용경유"].max(),
        "duplicate_dates": duplicate_dates,
    }])

    display(report)
    return report

In [ ]:
# ============================================================
# B-4-4. 통합 실행 함수
# ============================================================

def collect_preprocess_save_retail_avg(start_date, end_date=None, save_to_data_path=True):
    """
    전국 평균 소매가 수집 -> 현재 파일 형태 전처리 -> 저장 -> 검증.

    단일일:
      collect_preprocess_save_retail_avg("2026-06-09")

    기간:
      collect_preprocess_save_retail_avg("2008-01-01", "2026-06-10")
    """

    raw_retail, raw_bytes = collect_retail_avg_opinet_csv(
        start_date=start_date,
        end_date=end_date,
    )

    final_retail = preprocess_retail_avg_for_current_code(raw_retail)

    saved = save_retail_avg_outputs(
        raw_bytes=raw_bytes,
        final_retail=final_retail,
        start_date=start_date,
        end_date=end_date,
        save_to_data_path=save_to_data_path,
    )

    validate_retail_avg_current_shape(
        saved["final_data_path"] if save_to_data_path else saved["final_snapshot_path"]
    )

    return {
        "raw": raw_retail,
        "final": final_retail,
        "saved": saved,
    }

In [ ]:
# ============================================================
# B-4 실행
# ============================================================

if RUN_COLLECTION and RUN_RETAIL_AVG:
    RETAIL_AVG_RESULT = collect_preprocess_save_retail_avg(
        start_date=COLLECT_START_DATE,
        end_date=COLLECT_END_DATE,
        save_to_data_path=True,
    )
else:
    print("RUN_COLLECTION 또는 RUN_RETAIL_AVG가 False라서 전국 평균 소매가 수집을 실행하지 않았습니다.")

### 상표별 가격

In [ ]:
# ============================================================
# B-X. 상표별 가격 수집/전처리/저장
# ============================================================
from pathlib import Path
import io
import os
import re
import time
from datetime import datetime
from typing import Dict, Optional, Tuple

import pandas as pd
import requests


BRAND_PRICE_SOURCE_PAGE = "https://www.opinet.co.kr/user/dopostrm/dopOsTrmView.do"
BRAND_PRICE_CSV_URL = "https://www.opinet.co.kr/user/dopostrm/dopOsTrmCsv.do"

BRAND_PRICE_COLLECTION_DIR = Path(DATA_COLLECTION_PATH) / "brand_price"
BRAND_PRICE_RAW_DIR = BRAND_PRICE_COLLECTION_DIR / "raw"
BRAND_PRICE_FINAL_DIR = BRAND_PRICE_COLLECTION_DIR / "final"

BRAND_GASOLINE_FINAL_PATH = Path(DATA_PATH) / "brand_gasoline.csv"
BRAND_DIESEL_FINAL_PATH = Path(DATA_PATH) / "brand_diesel.csv"

for _dir in [BRAND_PRICE_COLLECTION_DIR, BRAND_PRICE_RAW_DIR, BRAND_PRICE_FINAL_DIR]:
    _dir.mkdir(parents=True, exist_ok=True)


BRAND_PRICE_PRODUCTS = {
    "gasoline": {"code": "B027", "fuel_name": "보통휘발유", "file_stem": "brand_gasoline"},
    "diesel": {"code": "D047", "fuel_name": "자동차용경유", "file_stem": "brand_diesel"},
}

BRAND_PRICE_COLUMNS = [
    "구분",
    "정유사평균",
    "SK에너지",
    "GS칼텍스",
    "HD현대오일뱅크",
    "S-OIL",
    "알뜰주유소",
    "(알뜰-자영)",
    "자가상표",
]

BRAND_PRICE_COLUMN_RENAME = {
    "정유사상표(전체)": "정유사평균",
    "알뜰주유소(전체)": "알뜰주유소",
    "알뜰평균": "알뜰주유소",
    "알뜰(자영)": "(알뜰-자영)",
}


def _brand_to_date(value):
    if value is None:
        raise ValueError("날짜가 비어 있습니다.")

    text = str(value).strip().replace("/", "-")
    if re.fullmatch(r"\d{8}", text):
        return datetime.strptime(text, "%Y%m%d").date()

    return pd.to_datetime(text).date()


def _brand_period_stamp(start_date, end_date=None) -> str:
    start = _brand_to_date(start_date)
    end = _brand_to_date(end_date or start_date)
    return f"{start:%Y%m%d}_{end:%Y%m%d}"


def _brand_extract_hidden(html: str, name: str, default: str = "") -> str:
    pattern = rf'name=["\']{re.escape(name)}["\'][^>]*value=["\']([^"\']*)'
    match = re.search(pattern, html)
    return match.group(1) if match else default


def _brand_date_parts(value) -> Tuple[str, str, str]:
    date_value = _brand_to_date(value)
    return f"{date_value:%Y}", f"{date_value:%m}", f"{date_value:%d}"


def _brand_read_csv_bytes(raw_bytes: bytes) -> pd.DataFrame:
    text = raw_bytes.decode("cp949", errors="replace").lstrip("\ufeff").strip()
    if not text.startswith("구분,"):
        raise RuntimeError(f"오피넷 상표별 CSV 응답 형식이 예상과 다릅니다.\n{text[:500]}")

    return pd.read_csv(io.BytesIO(raw_bytes), encoding="cp949")


def _brand_parse_kor_date_series(series: pd.Series) -> pd.Series:
    text = series.astype(str).str.replace("\xa0", "", regex=False).str.replace(" ", "", regex=False)
    extracted = text.str.extract(r"(?P<year>\d{2,4})년(?P<month>\d{1,2})월(?P<day>\d{1,2})일")

    if extracted.isna().any(axis=None):
        bad_values = series[extracted.isna().any(axis=1)].head(5).tolist()
        raise ValueError(f"날짜 파싱 실패 값: {bad_values}")

    years = extracted["year"].astype(int)
    years = years.where(years >= 100, years.where(years >= 50, years + 2000))
    years = years.where(years >= 100, years + 1900)

    return pd.to_datetime({
        "year": years,
        "month": extracted["month"].astype(int),
        "day": extracted["day"].astype(int),
    })


def collect_brand_price_opinet_csv(
    start_date: str,
    end_date: Optional[str] = None,
    product: str = "gasoline",
    timeout: Optional[int] = None,
    retry: Optional[int] = None,
    sleep_sec: Optional[float] = None,
) -> Tuple[pd.DataFrame, bytes]:
    """
    오피넷 평균판매가격 > 상표별 CSV를 수집합니다.
    product: "gasoline", "diesel", "B027", "D047" 지원
    start_date/end_date: 단일일 또는 기간 모두 지원
    """
    timeout = timeout or globals().get("REQUEST_TIMEOUT", 60)
    retry = retry or globals().get("REQUEST_RETRY", 3)
    sleep_sec = sleep_sec if sleep_sec is not None else globals().get("REQUEST_SLEEP_SEC", 0.5)

    product_info = BRAND_PRICE_PRODUCTS.get(product)
    if product_info is None:
        product_info = next(
            (info for info in BRAND_PRICE_PRODUCTS.values() if info["code"] == product),
            None,
        )
    if product_info is None:
        raise ValueError(f"지원하지 않는 product 값입니다: {product}")

    end_date = end_date or start_date

    sta_y, sta_m, sta_d = _brand_date_parts(start_date)
    end_y, end_m, end_d = _brand_date_parts(end_date)

    headers = dict(globals().get("HEADERS", {}))
    headers.update({
        "Referer": BRAND_PRICE_SOURCE_PAGE,
        "User-Agent": headers.get("User-Agent", "Mozilla/5.0"),
    })

    last_error = None
    for attempt in range(1, int(retry) + 1):
        try:
            session = requests.Session()
            page_response = session.get(BRAND_PRICE_SOURCE_PAGE, headers=headers, timeout=timeout)
            page_response.raise_for_status()
            page_response.encoding = "utf-8"
            html = page_response.text

            payload = {
                "all_chk_cnt": _brand_extract_hidden(html, "all_chk_cnt", "5"),
                "all_chk_roc_cnt": _brand_extract_hidden(html, "all_chk_roc_cnt", "10"),
                "INIF_FLAG": _brand_extract_hidden(html, "INIF_FLAG", "N"),
                "viewType": "POLL",
                "chk_cnt": _brand_extract_hidden(html, "chk_cnt", "4"),
                "chk_roc_cnt": "8",
                "h_maxYY": _brand_extract_hidden(html, "h_maxYY", end_y),
                "h_maxQQ": _brand_extract_hidden(html, "h_maxQQ", f"{end_y}1"),
                "h_maxMM": _brand_extract_hidden(html, "h_maxMM", f"{end_y}{end_m}"),
                "h_maxDD": _brand_extract_hidden(html, "h_maxDD", f"{end_y}{end_m}{end_d}"),
                "h_maxWW": _brand_extract_hidden(html, "h_maxWW", f"{end_y}{end_m}1"),
                "sta_dt": "",
                "end_dt": "",
                "TERM": "D",
                "STA_Y": sta_y,
                "STA_M": sta_m,
                "STA_D": sta_d,
                "STA_Q": str((int(sta_m) - 1) // 3 + 1),
                "STA_W": "1",
                "END_Y": end_y,
                "END_M": end_m,
                "END_D": end_d,
                "END_Q": str((int(end_m) - 1) // 3 + 1),
                "END_W": "1",
                "searchType": "POLL",
                "POLL_DIV_CD_REF": "Y",
                "POLL_DIV_CD_SKE": "Y",
                "POLL_DIV_CD_GSC": "Y",
                "POLL_DIV_CD_HDO": "Y",
                "POLL_DIV_CD_SOL": "Y",
                "POLL_DIV_CD_RTO": "Y",
                "POLL_DIV_CD_RTE": "Y",
                "POLL_DIV_CD_ETC": "Y",
                "sltProdCd": product_info["code"],
                "equal": "Y",
            }

            response = session.post(
                BRAND_PRICE_CSV_URL,
                data=payload,
                headers=headers,
                timeout=timeout,
            )
            response.raise_for_status()

            raw_bytes = response.content
            raw_df = _brand_read_csv_bytes(raw_bytes)
            return raw_df, raw_bytes

        except Exception as exc:
            last_error = exc
            if attempt < int(retry):
                time.sleep(float(sleep_sec) * attempt)

    raise RuntimeError(f"오피넷 상표별 가격 수집 실패: {last_error}")


def preprocess_brand_price_for_current_code(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    현재 01 전처리 코드 read_brand()가 기대하는 형태로 정리합니다.
    """
    df = raw_df.copy()
    df.columns = [str(c).strip().replace("\xa0", "") for c in df.columns]

    if "구분" not in df.columns:
        df = df.rename(columns={df.columns[0]: "구분"})

    df = df.rename(columns=BRAND_PRICE_COLUMN_RENAME)

    missing = [c for c in BRAND_PRICE_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"상표별 가격 필수 컬럼 누락: {missing}")

    df = df[BRAND_PRICE_COLUMNS].copy()
    df["_date"] = _brand_parse_kor_date_series(df["구분"])

    for col in BRAND_PRICE_COLUMNS:
        if col == "구분":
            continue
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
            .replace({"": pd.NA, "-": pd.NA, "nan": pd.NA, "None": pd.NA})
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = (
        df.sort_values("_date")
        .drop_duplicates("_date", keep="last")
        .reset_index(drop=True)
    )
    df["구분"] = df["_date"].dt.strftime("%Y년%m월%d일")

    return df[BRAND_PRICE_COLUMNS]


def save_brand_price_outputs(
    gasoline_df: pd.DataFrame,
    diesel_df: pd.DataFrame,
    raw_bytes_by_product: Dict[str, bytes],
    start_date: str,
    end_date: Optional[str] = None,
    save_to_data_path: bool = True,
) -> Dict[str, Path]:
    stamp = _brand_period_stamp(start_date, end_date or start_date)

    raw_gasoline_path = BRAND_PRICE_RAW_DIR / f"opinet_brand_gasoline_raw_{stamp}.csv"
    raw_diesel_path = BRAND_PRICE_RAW_DIR / f"opinet_brand_diesel_raw_{stamp}.csv"
    final_gasoline_snapshot_path = BRAND_PRICE_FINAL_DIR / f"brand_gasoline_{stamp}.csv"
    final_diesel_snapshot_path = BRAND_PRICE_FINAL_DIR / f"brand_diesel_{stamp}.csv"

    raw_gasoline_path.write_bytes(raw_bytes_by_product["gasoline"])
    raw_diesel_path.write_bytes(raw_bytes_by_product["diesel"])

    gasoline_df.to_csv(final_gasoline_snapshot_path, index=False, encoding="utf-8-sig")
    diesel_df.to_csv(final_diesel_snapshot_path, index=False, encoding="utf-8-sig")

    paths = {
        "raw_gasoline_path": raw_gasoline_path,
        "raw_diesel_path": raw_diesel_path,
        "final_gasoline_snapshot_path": final_gasoline_snapshot_path,
        "final_diesel_snapshot_path": final_diesel_snapshot_path,
    }

    if save_to_data_path:
        gasoline_df.to_csv(BRAND_GASOLINE_FINAL_PATH, index=False, encoding="utf-8-sig")
        diesel_df.to_csv(BRAND_DIESEL_FINAL_PATH, index=False, encoding="utf-8-sig")
        paths["final_gasoline_data_path"] = BRAND_GASOLINE_FINAL_PATH
        paths["final_diesel_data_path"] = BRAND_DIESEL_FINAL_PATH

    return paths


def validate_brand_price_current_shape(path, fuel_name: str) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")
    missing = [c for c in BRAND_PRICE_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"{fuel_name} 상표별 가격 컬럼 누락: {missing}")

    dates = _brand_parse_kor_date_series(df["구분"])
    numeric_cols = [c for c in BRAND_PRICE_COLUMNS if c != "구분"]
    numeric_df = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

    summary = pd.DataFrame([{
        "path": str(path),
        "fuel": fuel_name,
        "rows": len(df),
        "date_min": dates.min().date(),
        "date_max": dates.max().date(),
        "duplicate_dates": int(dates.duplicated().sum()),
        "numeric_columns": ", ".join(numeric_cols),
        "all_null_numeric_rows": int(numeric_df.isna().all(axis=1).sum()),
    }])

    display(summary)
    return summary


def collect_preprocess_save_brand_prices(
    start_date: str = None,
    end_date: Optional[str] = None,
    save_to_data_path: bool = True,
) -> Dict[str, object]:
    start_date = start_date or globals().get("COLLECT_START_DATE", "2008-01-01")
    end_date = end_date or globals().get("COLLECT_END_DATE", start_date)

    raw_gasoline_df, raw_gasoline_bytes = collect_brand_price_opinet_csv(
        start_date=start_date,
        end_date=end_date,
        product="gasoline",
    )
    raw_diesel_df, raw_diesel_bytes = collect_brand_price_opinet_csv(
        start_date=start_date,
        end_date=end_date,
        product="diesel",
    )

    gasoline_df = preprocess_brand_price_for_current_code(raw_gasoline_df)
    diesel_df = preprocess_brand_price_for_current_code(raw_diesel_df)

    paths = save_brand_price_outputs(
        gasoline_df=gasoline_df,
        diesel_df=diesel_df,
        raw_bytes_by_product={
            "gasoline": raw_gasoline_bytes,
            "diesel": raw_diesel_bytes,
        },
        start_date=start_date,
        end_date=end_date,
        save_to_data_path=save_to_data_path,
    )

    print("[상표별 가격 저장 완료]")
    for key, path in paths.items():
        print(f"- {key}: {path}")
    print(f"- gasoline_rows: {len(gasoline_df):,}")
    print(f"- diesel_rows: {len(diesel_df):,}")
    print(f"- gasoline_date_min: {gasoline_df['구분'].iloc[0]}")
    print(f"- gasoline_date_max: {gasoline_df['구분'].iloc[-1]}")
    print(f"- diesel_date_min: {diesel_df['구분'].iloc[0]}")
    print(f"- diesel_date_max: {diesel_df['구분'].iloc[-1]}")

    gasoline_summary = validate_brand_price_current_shape(
        paths["final_gasoline_data_path"] if save_to_data_path else paths["final_gasoline_snapshot_path"],
        "보통휘발유",
    )
    diesel_summary = validate_brand_price_current_shape(
        paths["final_diesel_data_path"] if save_to_data_path else paths["final_diesel_snapshot_path"],
        "자동차용경유",
    )

    return {
        "gasoline": gasoline_df,
        "diesel": diesel_df,
        "paths": paths,
        "gasoline_summary": gasoline_summary,
        "diesel_summary": diesel_summary,
    }


if globals().get("RUN_COLLECTION", False) and globals().get("RUN_BRAND_PRICE", True):
    brand_price_result = collect_preprocess_save_brand_prices(
        start_date=COLLECT_START_DATE,
        end_date=COLLECT_END_DATE,
        save_to_data_path=True,
    )
else:
    print("[상표별 가격] RUN_COLLECTION 또는 RUN_BRAND_PRICE가 False라 실행하지 않았습니다.")

### 유류세 변동

In [ ]:
# ============================================================
# B-X. 유류세 변동 수집/전처리/저장
# ============================================================
from pathlib import Path
import io
import os
import re
import time
from datetime import datetime
from html import escape
from typing import Dict, Optional, Tuple

import pandas as pd
import requests


FUEL_TAX_SELECT_URL = "https://www.opinet.co.kr/user/oftvat/getOfttexSelect.do"
FUEL_TAX_PRINT_TIME_URL = "https://www.opinet.co.kr/user/oftvat/getOfttexPrintTime.do"

FUEL_TAX_COLLECTION_DIR = Path(DATA_COLLECTION_PATH) / "fuel_tax_trend"
FUEL_TAX_RAW_DIR = FUEL_TAX_COLLECTION_DIR / "raw"
FUEL_TAX_FINAL_DIR = FUEL_TAX_COLLECTION_DIR / "final"

GASOLINE_TAX_TREND_FINAL_PATH = Path(DATA_PATH) / "gasoline_tax_trend.xls"
DIESEL_TAX_TREND_FINAL_PATH = Path(DATA_PATH) / "diesel_tax_trend.xls"

for _dir in [FUEL_TAX_COLLECTION_DIR, FUEL_TAX_RAW_DIR, FUEL_TAX_FINAL_DIR]:
    _dir.mkdir(parents=True, exist_ok=True)


FUEL_TAX_PRODUCTS = {
    "gasoline": {
        "opinet_code": "CC",
        "fuel_name": "보통휘발유",
        "file_name": "gasoline_tax_trend.xls",
        "final_path": GASOLINE_TAX_TREND_FINAL_PATH,
    },
    "diesel": {
        "opinet_code": "DD",
        "fuel_name": "자동차용경유",
        "file_name": "diesel_tax_trend.xls",
        "final_path": DIESEL_TAX_TREND_FINAL_PATH,
    },
}

FUEL_TAX_COLUMNS = [
    "변동일자",
    "개별소비세",
    "교통에너지환경세",
    "교육세",
    "주행세",
    "합계",
    "판매부과금",
]


def _fuel_tax_to_date(value):
    text = str(value).strip().replace("/", "-")
    if re.fullmatch(r"\d{8}", text):
        return datetime.strptime(text, "%Y%m%d").date()
    return pd.to_datetime(text).date()


def _fuel_tax_date_parts(value) -> Tuple[str, str, str]:
    date_value = _fuel_tax_to_date(value)
    return f"{date_value:%Y}", f"{date_value:%m}", f"{date_value:%d}"


def _fuel_tax_period_stamp(start_date, end_date=None) -> str:
    start = _fuel_tax_to_date(start_date)
    end = _fuel_tax_to_date(end_date or start_date)
    return f"{start:%Y%m%d}_{end:%Y%m%d}"


def _fuel_tax_parse_kor_date_series(series: pd.Series) -> pd.Series:
    text = series.astype(str).str.replace("\xa0", "", regex=False).str.replace(" ", "", regex=False)
    extracted = text.str.extract(r"(?P<year>\d{2,4})년(?P<month>\d{1,2})월(?P<day>\d{1,2})일")

    if extracted.isna().any(axis=None):
        bad_values = series[extracted.isna().any(axis=1)].head(5).tolist()
        raise ValueError(f"날짜 파싱 실패 값: {bad_values}")

    years = extracted["year"].astype(int)
    years = years.where(years >= 100, years.where(years >= 50, years + 2000))
    years = years.where(years >= 100, years + 1900)

    return pd.to_datetime({
        "year": years,
        "month": extracted["month"].astype(int),
        "day": extracted["day"].astype(int),
    })


def _open_fuel_tax_session(timeout: Optional[int] = None) -> requests.Session:
    timeout = timeout or globals().get("REQUEST_TIMEOUT", 60)

    headers = dict(globals().get("HEADERS", {}))
    headers.update({"User-Agent": headers.get("User-Agent", "Mozilla/5.0")})

    session = requests.Session()
    gate_response = session.get(FUEL_TAX_SELECT_URL, headers=headers, timeout=timeout)
    gate_response.raise_for_status()
    gate_response.encoding = "utf-8"

    match = re.search(r"frm\.opinet_key\.value\s*=\s*'([^']+)'", gate_response.text)
    if match:
        open_response = session.post(
            FUEL_TAX_SELECT_URL,
            data={"opinet_key": match.group(1), "netfunnel_key": ""},
            headers=headers,
            timeout=timeout,
        )
        open_response.raise_for_status()

    return session


def _flatten_fuel_tax_column(col) -> str:
    if isinstance(col, tuple):
        parts = [
            str(x).replace("\xa0", "").strip()
            for x in col
            if str(x).strip()
            and not str(x).startswith("Unnamed")
            and str(x).strip() != "유류세"
        ]
        if parts:
            return parts[-1]
    return str(col).replace("\xa0", "").strip()


def _parse_fuel_tax_html_table(html: str) -> pd.DataFrame:
    tables = pd.read_html(io.StringIO(html), header=[0, 1])

    parsed = None
    for table in tables:
        tmp = table.copy()
        tmp.columns = [_flatten_fuel_tax_column(c) for c in tmp.columns]
        if "변동일자" in tmp.columns:
            parsed = tmp
            break

    if parsed is None:
        raise RuntimeError("오피넷 유류세 표에서 '변동일자' 컬럼을 찾지 못했습니다.")

    missing = [c for c in FUEL_TAX_COLUMNS if c not in parsed.columns]
    if missing:
        raise ValueError(f"오피넷 유류세 표 필수 컬럼 누락: {missing}")

    return parsed[FUEL_TAX_COLUMNS].copy()


def collect_fuel_tax_trend_opinet_html(
    start_date: str,
    end_date: Optional[str] = None,
    fuel: str = "gasoline",
    source_start_date: str = "1997-01-01",
    timeout: Optional[int] = None,
    retry: Optional[int] = None,
    sleep_sec: Optional[float] = None,
) -> Tuple[pd.DataFrame, str]:
    """
    오피넷 유류세 > 세금변동추이 > 기간별 표를 수집합니다.

    start_date/end_date는 저장 스냅샷 기준 기간입니다.
    유류세는 변동 이벤트 표이므로, 전처리 ffill 기준을 보존하기 위해
    실제 원천 조회는 source_start_date부터 end_date까지 수행합니다.
    """
    timeout = timeout or globals().get("REQUEST_TIMEOUT", 60)
    retry = retry or globals().get("REQUEST_RETRY", 3)
    sleep_sec = sleep_sec if sleep_sec is not None else globals().get("REQUEST_SLEEP_SEC", 0.5)

    if fuel not in FUEL_TAX_PRODUCTS:
        raise ValueError(f"지원하지 않는 fuel 값입니다: {fuel}")

    product = FUEL_TAX_PRODUCTS[fuel]
    end_date = end_date or start_date

    query_start = source_start_date
    query_end = end_date

    start_yy, start_mm, start_dd = _fuel_tax_date_parts(query_start)
    end_yy, end_mm, end_dd = _fuel_tax_date_parts(query_end)

    headers = dict(globals().get("HEADERS", {}))
    headers.update({
        "Referer": FUEL_TAX_SELECT_URL,
        "User-Agent": headers.get("User-Agent", "Mozilla/5.0"),
    })

    payload = {
        "TERM": "T",
        "start_yy": start_yy,
        "start_mm": start_mm,
        "start_dd": start_dd,
        "end_yy": end_yy,
        "end_mm": end_mm,
        "end_dd": end_dd,
        "start2_yy": start_yy,
        "start2_mm": start_mm,
        "start2_dd": start_dd,
        "KNOC_TERM1": product["opinet_code"],
        "KNOC_TERM": "A",
    }

    last_error = None
    for attempt in range(1, int(retry) + 1):
        try:
            session = _open_fuel_tax_session(timeout=timeout)
            response = session.post(
                FUEL_TAX_PRINT_TIME_URL,
                data=payload,
                headers=headers,
                timeout=timeout,
            )
            response.raise_for_status()
            response.encoding = "utf-8"
            html = response.text

            if "변동일자" not in html:
                raise RuntimeError(f"오피넷 유류세 응답에 표가 없습니다.\n{html[:500]}")

            if product["fuel_name"] not in html:
                raise RuntimeError(f"오피넷 유류세 응답의 제품명이 예상과 다릅니다: {product['fuel_name']}")

            raw_df = _parse_fuel_tax_html_table(html)
            return raw_df, html

        except Exception as exc:
            last_error = exc
            if attempt < int(retry):
                time.sleep(float(sleep_sec) * attempt)

    raise RuntimeError(f"오피넷 유류세 변동 수집 실패: {last_error}")


def preprocess_fuel_tax_trend_for_current_code(
    raw_df: pd.DataFrame,
    end_date: str,
) -> pd.DataFrame:
    """
    현재 01 전처리 코드 read_tax()가 기대하는 이벤트 표 형태로 정리합니다.
    """
    df = raw_df.copy()
    df.columns = [str(c).replace("\xa0", "").strip() for c in df.columns]

    missing = [c for c in FUEL_TAX_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"유류세 필수 컬럼 누락: {missing}")

    df = df[FUEL_TAX_COLUMNS].copy()
    df["_date"] = _fuel_tax_parse_kor_date_series(df["변동일자"])

    end_ts = pd.Timestamp(_fuel_tax_to_date(end_date))
    df = df[df["_date"] <= end_ts].copy()

    for col in FUEL_TAX_COLUMNS:
        if col == "변동일자":
            continue
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
            .replace({"nan": "", "None": ""})
        )

    df = (
        df.sort_values("_date")
        .drop_duplicates("_date", keep="last")
        .reset_index(drop=True)
    )
    df["변동일자"] = df["_date"].dt.strftime("%Y년%m월%d일")

    return df[FUEL_TAX_COLUMNS]


def _fuel_tax_format_cell(value, numeric: bool = False) -> str:
    if pd.isna(value):
        return ""

    text = str(value).strip()
    if text in ["", "nan", "None"]:
        return ""

    if numeric and text != "-":
        num = pd.to_numeric(text, errors="coerce")
        if pd.notna(num):
            return f"{float(num):.2f}"

    return text


def _fuel_tax_to_current_xls_html(df: pd.DataFrame, title: str) -> str:
    """
    .xls 확장자로 저장하지만 내용은 HTML table입니다.
    현재 read_excel_robust()의 HTML fallback과 read_tax()의
    df.columns = df.iloc[0] 로직에 맞추기 위해 헤더 행을 2번 둡니다.
    """
    header_html = "".join(f"<th>{escape(c)}</th>" for c in FUEL_TAX_COLUMNS)
    rows = [
        f"<tr>{header_html}</tr>",
        "<tr>" + "".join(f"<td>{escape(c)}</td>" for c in FUEL_TAX_COLUMNS) + "</tr>",
    ]

    for _, row in df.iterrows():
        cells = []
        for col in FUEL_TAX_COLUMNS:
            value = _fuel_tax_format_cell(row[col], numeric=(col != "변동일자"))
            cells.append(f"<td>{escape(value)}</td>")
        rows.append("<tr>" + "".join(cells) + "</tr>")

    return f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>{escape(title)}</title>
</head>
<body>
<table>
{chr(10).join(rows)}
</table>
</body>
</html>
"""


def save_fuel_tax_trend_outputs(
    gasoline_df: pd.DataFrame,
    diesel_df: pd.DataFrame,
    raw_html_by_fuel: Dict[str, str],
    start_date: str,
    end_date: Optional[str] = None,
    save_to_data_path: bool = True,
) -> Dict[str, Path]:
    stamp = _fuel_tax_period_stamp(start_date, end_date or start_date)

    raw_gasoline_path = FUEL_TAX_RAW_DIR / f"opinet_gasoline_tax_trend_raw_{stamp}.html"
    raw_diesel_path = FUEL_TAX_RAW_DIR / f"opinet_diesel_tax_trend_raw_{stamp}.html"

    final_gasoline_snapshot_path = FUEL_TAX_FINAL_DIR / f"gasoline_tax_trend_{stamp}.xls"
    final_diesel_snapshot_path = FUEL_TAX_FINAL_DIR / f"diesel_tax_trend_{stamp}.xls"

    raw_gasoline_path.write_text(raw_html_by_fuel["gasoline"], encoding="utf-8")
    raw_diesel_path.write_text(raw_html_by_fuel["diesel"], encoding="utf-8")

    gasoline_xls_html = _fuel_tax_to_current_xls_html(gasoline_df, "gasoline_tax_trend")
    diesel_xls_html = _fuel_tax_to_current_xls_html(diesel_df, "diesel_tax_trend")

    final_gasoline_snapshot_path.write_text(gasoline_xls_html, encoding="utf-8-sig")
    final_diesel_snapshot_path.write_text(diesel_xls_html, encoding="utf-8-sig")

    paths = {
        "raw_gasoline_path": raw_gasoline_path,
        "raw_diesel_path": raw_diesel_path,
        "final_gasoline_snapshot_path": final_gasoline_snapshot_path,
        "final_diesel_snapshot_path": final_diesel_snapshot_path,
    }

    if save_to_data_path:
        GASOLINE_TAX_TREND_FINAL_PATH.write_text(gasoline_xls_html, encoding="utf-8-sig")
        DIESEL_TAX_TREND_FINAL_PATH.write_text(diesel_xls_html, encoding="utf-8-sig")
        paths["final_gasoline_data_path"] = GASOLINE_TAX_TREND_FINAL_PATH
        paths["final_diesel_data_path"] = DIESEL_TAX_TREND_FINAL_PATH

    return paths


def read_saved_fuel_tax_trend_like_current_code(path) -> pd.DataFrame:
    tables = pd.read_html(path, header=0)
    if not tables:
        raise ValueError(f"HTML table을 찾지 못했습니다: {path}")

    df = tables[0]
    df.columns = df.iloc[0]
    df = df.iloc[1:].copy().reset_index(drop=True)
    return df


def validate_fuel_tax_trend_current_shape(
    path,
    fuel_name: str,
    requested_start_date: str,
    requested_end_date: str,
) -> pd.DataFrame:
    df = read_saved_fuel_tax_trend_like_current_code(path)

    missing = [c for c in FUEL_TAX_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"{fuel_name} 유류세 변동 컬럼 누락: {missing}")

    dates = _fuel_tax_parse_kor_date_series(df["변동일자"])
    req_start = pd.Timestamp(_fuel_tax_to_date(requested_start_date))
    req_end = pd.Timestamp(_fuel_tax_to_date(requested_end_date))

    numeric_cols = [c for c in FUEL_TAX_COLUMNS if c != "변동일자"]
    numeric_df = (
        df[numeric_cols]
        .replace("-", pd.NA)
        .apply(pd.to_numeric, errors="coerce")
    )

    summary = pd.DataFrame([{
        "path": str(path),
        "fuel": fuel_name,
        "rows": len(df),
        "date_min": dates.min().date(),
        "date_max": dates.max().date(),
        "duplicate_dates": int(dates.duplicated().sum()),
        "changes_in_requested_period": int(dates.between(req_start, req_end).sum()),
        "all_null_numeric_rows": int(numeric_df.isna().all(axis=1).sum()),
        "columns": ", ".join(df.columns.astype(str).tolist()),
    }])

    display(summary)
    return summary


def collect_preprocess_save_fuel_tax_trends(
    start_date: str = None,
    end_date: Optional[str] = None,
    save_to_data_path: bool = True,
) -> Dict[str, object]:
    start_date = start_date or globals().get("COLLECT_START_DATE", "2008-01-01")
    end_date = end_date or globals().get("COLLECT_END_DATE", start_date)

    raw_gasoline_df, raw_gasoline_html = collect_fuel_tax_trend_opinet_html(
        start_date=start_date,
        end_date=end_date,
        fuel="gasoline",
    )
    raw_diesel_df, raw_diesel_html = collect_fuel_tax_trend_opinet_html(
        start_date=start_date,
        end_date=end_date,
        fuel="diesel",
    )

    gasoline_df = preprocess_fuel_tax_trend_for_current_code(
        raw_gasoline_df,
        end_date=end_date,
    )
    diesel_df = preprocess_fuel_tax_trend_for_current_code(
        raw_diesel_df,
        end_date=end_date,
    )

    paths = save_fuel_tax_trend_outputs(
        gasoline_df=gasoline_df,
        diesel_df=diesel_df,
        raw_html_by_fuel={
            "gasoline": raw_gasoline_html,
            "diesel": raw_diesel_html,
        },
        start_date=start_date,
        end_date=end_date,
        save_to_data_path=save_to_data_path,
    )

    print("[유류세 변동 저장 완료]")
    for key, path in paths.items():
        print(f"- {key}: {path}")
    print(f"- gasoline_rows: {len(gasoline_df):,}")
    print(f"- diesel_rows: {len(diesel_df):,}")
    print(f"- gasoline_date_min: {gasoline_df['변동일자'].iloc[0]}")
    print(f"- gasoline_date_max: {gasoline_df['변동일자'].iloc[-1]}")
    print(f"- diesel_date_min: {diesel_df['변동일자'].iloc[0]}")
    print(f"- diesel_date_max: {diesel_df['변동일자'].iloc[-1]}")

    gasoline_summary = validate_fuel_tax_trend_current_shape(
        paths["final_gasoline_data_path"] if save_to_data_path else paths["final_gasoline_snapshot_path"],
        "보통휘발유",
        requested_start_date=start_date,
        requested_end_date=end_date,
    )
    diesel_summary = validate_fuel_tax_trend_current_shape(
        paths["final_diesel_data_path"] if save_to_data_path else paths["final_diesel_snapshot_path"],
        "자동차용경유",
        requested_start_date=start_date,
        requested_end_date=end_date,
    )

    return {
        "gasoline": gasoline_df,
        "diesel": diesel_df,
        "paths": paths,
        "gasoline_summary": gasoline_summary,
        "diesel_summary": diesel_summary,
    }


if globals().get("RUN_COLLECTION", False) and globals().get("RUN_FUEL_TAX_TREND", True):
    fuel_tax_trend_result = collect_preprocess_save_fuel_tax_trends(
        start_date=COLLECT_START_DATE,
        end_date=COLLECT_END_DATE,
        save_to_data_path=True,
    )
else:
    print("[유류세 변동] RUN_COLLECTION 또는 RUN_FUEL_TAX_TREND가 False라 실행하지 않았습니다.")

### 정유사 주산 공급가격

In [ ]:
# ============================================================
# B-X. 정유사 주간 공급가격 수집/전처리/저장
# ============================================================
from pathlib import Path
import io
import os
import re
import time
import calendar
from datetime import datetime
from typing import Optional, Tuple

import pandas as pd
import requests


REFINERY_SUPPLY_SOURCE_PAGE = "https://www.opinet.co.kr/user/dopdavcow/dopAvcowSelect.do"
REFINERY_SUPPLY_CSV_URL = "https://www.opinet.co.kr/user/dopavcow/dopAvcowCsv.do"

REFINERY_SUPPLY_COLLECTION_DIR = Path(DATA_COLLECTION_PATH) / "refinery_weekly_supply"
REFINERY_SUPPLY_RAW_DIR = REFINERY_SUPPLY_COLLECTION_DIR / "raw"
REFINERY_SUPPLY_FINAL_DIR = REFINERY_SUPPLY_COLLECTION_DIR / "final"

REFINERY_SUPPLY_FINAL_PATH = Path(DATA_PATH) / "refinery_weekly_supply_prices_by_product.csv"

for _dir in [REFINERY_SUPPLY_COLLECTION_DIR, REFINERY_SUPPLY_RAW_DIR, REFINERY_SUPPLY_FINAL_DIR]:
    _dir.mkdir(parents=True, exist_ok=True)

REFINERY_SUPPLY_COLUMNS = ["구분", "보통휘발유", "자동차용경유"]


def _refinery_to_date(value):
    text = str(value).strip().replace("/", "-")
    if re.fullmatch(r"\d{8}", text):
        return datetime.strptime(text, "%Y%m%d").date()
    return pd.to_datetime(text).date()


def _refinery_period_stamp(start_date, end_date=None) -> str:
    start = _refinery_to_date(start_date)
    end = _refinery_to_date(end_date or start_date)
    return f"{start:%Y%m%d}_{end:%Y%m%d}"


def _refinery_date_to_opinet_week(value) -> Tuple[str, str, str]:
    """
    현재 read_refinery()의 주차 해석과 맞춤.
    1주=1~7일, 2주=8~14일, 3주=15~21일, 4주=22~28일, 5주=29일~월말
    """
    dt = _refinery_to_date(value)
    week = min(((dt.day - 1) // 7) + 1, 5)
    return f"{dt:%Y}", f"{dt:%m}", str(week)


def _refinery_extract_hidden(html: str, name: str, default: str = "") -> str:
    pattern = rf'name=["\']{re.escape(name)}["\'][^>]*value=["\']([^"\']*)'
    match = re.search(pattern, html)
    return match.group(1) if match else default


def _decode_refinery_csv(raw_bytes: bytes) -> str:
    last_error = None
    for enc in ["euc-kr", "cp949", "utf-8-sig", "utf-8"]:
        try:
            text = raw_bytes.decode(enc)
            if "구분" in text:
                return text
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"정유사 공급가격 CSV 디코딩 실패: {last_error}")


def _read_refinery_csv_robust(path) -> pd.DataFrame:
    last_error = None
    for enc in ["utf-8-sig", "utf-8", "euc-kr", "cp949"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as exc:
            last_error = exc
    raise last_error


def _parse_refinery_week_end_series(series: pd.Series) -> pd.Series:
    text = series.astype(str).str.replace("\xa0", "", regex=False).str.replace(" ", "", regex=False)
    extracted = text.str.extract(r"(?P<year>\d{2})년(?P<month>\d{2})월(?P<week>\d)주")

    if extracted.isna().any(axis=None):
        bad_values = series[extracted.isna().any(axis=1)].head(5).tolist()
        raise ValueError(f"주차 파싱 실패 값: {bad_values}")

    years = extracted["year"].astype(int)
    years = years.where(years >= 50, years + 2000)
    years = years.where(years >= 100, years + 1900)

    months = extracted["month"].astype(int)
    weeks = extracted["week"].astype(int)

    dates = []
    for year, month, week in zip(years, months, weeks):
        last_day = calendar.monthrange(int(year), int(month))[1]
        if week == 1:
            day = min(7, last_day)
        elif week == 2:
            day = min(14, last_day)
        elif week == 3:
            day = min(21, last_day)
        elif week == 4:
            day = min(28, last_day)
        elif week == 5:
            day = last_day
        else:
            raise ValueError(f"주차 값이 올바르지 않습니다: {week}")
        dates.append(pd.Timestamp(int(year), int(month), int(day)))

    return pd.Series(dates, index=series.index)


def collect_refinery_weekly_supply_opinet_csv(
    start_date: str,
    end_date: Optional[str] = None,
    timeout: Optional[int] = None,
    retry: Optional[int] = None,
    sleep_sec: Optional[float] = None,
) -> Tuple[pd.DataFrame, bytes]:
    """
    오피넷 정유사 > 주간공급가격 > 제품별 CSV를 세전 기준으로 수집합니다.
    start_date/end_date는 날짜로 받되, 오피넷 조회는 해당 날짜가 속한 월-주차로 변환합니다.
    단일일도 같은 주 1행으로 수집됩니다.
    """
    timeout = timeout or globals().get("REQUEST_TIMEOUT", 60)
    retry = retry or globals().get("REQUEST_RETRY", 3)
    sleep_sec = sleep_sec if sleep_sec is not None else globals().get("REQUEST_SLEEP_SEC", 0.5)

    end_date = end_date or start_date

    sta_y, sta_m, sta_w = _refinery_date_to_opinet_week(start_date)
    end_y, end_m, end_w = _refinery_date_to_opinet_week(end_date)

    headers = dict(globals().get("HEADERS", {}))
    headers.update({
        "Referer": REFINERY_SUPPLY_SOURCE_PAGE,
        "User-Agent": headers.get("User-Agent", "Mozilla/5.0"),
    })

    last_error = None
    for attempt in range(1, int(retry) + 1):
        try:
            session = requests.Session()
            page_response = session.get(REFINERY_SUPPLY_SOURCE_PAGE, headers=headers, timeout=timeout)
            page_response.raise_for_status()
            page_response.encoding = "utf-8"
            html = page_response.text

            payload = {
                "all_chk_cnt": _refinery_extract_hidden(html, "all_chk_cnt", "5"),
                "INIF_FLAG": _refinery_extract_hidden(html, "INIF_FLAG", "N"),
                "chk_cnt": "2",
                "h_default_y": _refinery_extract_hidden(html, "h_default_y", end_y),
                "h_default_q": _refinery_extract_hidden(html, "h_default_q", f"{end_y}1"),
                "h_default_m": _refinery_extract_hidden(html, "h_default_m", f"{end_y}{end_m}"),
                "h_default_w": _refinery_extract_hidden(html, "h_default_w", f"{end_y}{end_m}{end_w}"),
                "TERM": "W",
                "STA_Y": sta_y,
                "STA_M": sta_m,
                "STA_W": sta_w,
                "STA_Q": str((int(sta_m) - 1) // 3 + 1),
                "END_Y": end_y,
                "END_M": end_m,
                "END_W": end_w,
                "END_Q": str((int(end_m) - 1) // 3 + 1),
                "tgubun": "b",          # b=세전, a=세후
                "OIL_CD_B027": "Y",     # 보통휘발유
                "OIL_CD_D047": "Y",     # 자동차용경유
                "equal": "N",           # 단일 주도 주차 1행 형태로 받기
            }

            response = session.post(
                REFINERY_SUPPLY_CSV_URL,
                data=payload,
                headers=headers,
                timeout=timeout,
            )
            response.raise_for_status()

            raw_bytes = response.content
            text = _decode_refinery_csv(raw_bytes)
            if not text.lstrip("\ufeff").startswith("구분,"):
                raise RuntimeError(f"오피넷 정유사 공급가격 CSV 응답 형식이 예상과 다릅니다.\n{text[:500]}")

            raw_df = pd.read_csv(io.StringIO(text))
            return raw_df, raw_bytes

        except Exception as exc:
            last_error = exc
            if attempt < int(retry):
                time.sleep(float(sleep_sec) * attempt)

    raise RuntimeError(f"오피넷 정유사 주간 공급가격 수집 실패: {last_error}")


def preprocess_refinery_weekly_supply_for_current_code(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    현재 01 전처리 코드 read_refinery()가 기대하는 형태로 정리합니다.
    """
    df = raw_df.copy()
    df.columns = [str(c).strip().replace("\xa0", "") for c in df.columns]
    df = df.rename(columns={"일반휘발유": "보통휘발유"})

    missing = [c for c in REFINERY_SUPPLY_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"정유사 주간 공급가격 필수 컬럼 누락: {missing}")

    df = df[REFINERY_SUPPLY_COLUMNS].copy()
    df["구분"] = df["구분"].astype(str).str.replace("\xa0", "", regex=False).str.replace(" ", "", regex=False).str.strip()

    # 주차 행만 유지
    df = df[df["구분"].str.match(r"^\d{2}년\d{2}월\d주$")].copy()
    df["_date"] = _parse_refinery_week_end_series(df["구분"])

    for col in ["보통휘발유", "자동차용경유"]:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
            .replace({"": pd.NA, "-": pd.NA, "nan": pd.NA, "None": pd.NA})
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = (
        df.sort_values("_date")
        .drop_duplicates("구분", keep="last")
        .reset_index(drop=True)
    )

    return df[REFINERY_SUPPLY_COLUMNS]


def save_refinery_weekly_supply_outputs(
    final_df: pd.DataFrame,
    raw_bytes: bytes,
    start_date: str,
    end_date: Optional[str] = None,
    save_to_data_path: bool = True,
) -> dict:
    stamp = _refinery_period_stamp(start_date, end_date or start_date)

    raw_path = REFINERY_SUPPLY_RAW_DIR / f"opinet_refinery_weekly_supply_raw_{stamp}.csv"
    final_snapshot_path = REFINERY_SUPPLY_FINAL_DIR / f"refinery_weekly_supply_prices_by_product_{stamp}.csv"

    raw_path.write_bytes(raw_bytes)
    final_df.to_csv(final_snapshot_path, index=False, encoding="utf-8-sig")

    paths = {
        "raw_path": raw_path,
        "final_snapshot_path": final_snapshot_path,
    }

    if save_to_data_path:
        REFINERY_SUPPLY_FINAL_PATH.parent.mkdir(parents=True, exist_ok=True)
        final_df.to_csv(REFINERY_SUPPLY_FINAL_PATH, index=False, encoding="utf-8-sig")
        paths["final_data_path"] = REFINERY_SUPPLY_FINAL_PATH

    return paths


def validate_refinery_weekly_supply_current_shape(path) -> pd.DataFrame:
    df = _read_refinery_csv_robust(path)

    missing = [c for c in REFINERY_SUPPLY_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"정유사 주간 공급가격 컬럼 누락: {missing}")

    dates = _parse_refinery_week_end_series(df["구분"])
    numeric_df = df[["보통휘발유", "자동차용경유"]].apply(pd.to_numeric, errors="coerce")

    summary = pd.DataFrame([{
        "path": str(path),
        "rows": len(df),
        "week_min": df["구분"].iloc[0] if len(df) else None,
        "week_max": df["구분"].iloc[-1] if len(df) else None,
        "date_min": dates.min().date() if len(df) else None,
        "date_max": dates.max().date() if len(df) else None,
        "duplicate_weeks": int(df["구분"].duplicated().sum()),
        "gasoline_min": numeric_df["보통휘발유"].min(),
        "gasoline_max": numeric_df["보통휘발유"].max(),
        "diesel_min": numeric_df["자동차용경유"].min(),
        "diesel_max": numeric_df["자동차용경유"].max(),
        "all_null_numeric_rows": int(numeric_df.isna().all(axis=1).sum()),
    }])

    display(summary)
    return summary


def collect_preprocess_save_refinery_weekly_supply(
    start_date: str = None,
    end_date: Optional[str] = None,
    save_to_data_path: bool = True,
) -> dict:
    start_date = start_date or globals().get("COLLECT_START_DATE", "2008-01-01")
    end_date = end_date or globals().get("COLLECT_END_DATE", start_date)

    raw_df, raw_bytes = collect_refinery_weekly_supply_opinet_csv(
        start_date=start_date,
        end_date=end_date,
    )

    final_df = preprocess_refinery_weekly_supply_for_current_code(raw_df)

    paths = save_refinery_weekly_supply_outputs(
        final_df=final_df,
        raw_bytes=raw_bytes,
        start_date=start_date,
        end_date=end_date,
        save_to_data_path=save_to_data_path,
    )

    print("[정유사 주간 공급가격 저장 완료]")
    for key, path in paths.items():
        print(f"- {key}: {path}")
    print(f"- rows: {len(final_df):,}")
    print(f"- week_min: {final_df['구분'].iloc[0] if len(final_df) else None}")
    print(f"- week_max: {final_df['구분'].iloc[-1] if len(final_df) else None}")

    summary = validate_refinery_weekly_supply_current_shape(
        paths["final_data_path"] if save_to_data_path else paths["final_snapshot_path"]
    )

    return {
        "final": final_df,
        "paths": paths,
        "summary": summary,
    }


if globals().get("RUN_COLLECTION", False) and globals().get("RUN_REFINERY_WEEKLY_SUPPLY", True):
    refinery_weekly_supply_result = collect_preprocess_save_refinery_weekly_supply(
        start_date=COLLECT_START_DATE,
        end_date=COLLECT_END_DATE,
        save_to_data_path=True,
    )
else:
    print("[정유사 주간 공급가격] RUN_COLLECTION 또는 RUN_REFINERY_WEEKLY_SUPPLY가 False라 실행하지 않았습니다.")

### 지역별 주유소

In [ ]:
# ============================================================
# 06-B 전국 개별 주유소/지역별 주유소 원자료 - 설치/경로 설정
# ============================================================
!apt-get update -y
!apt-get install -y wget gnupg unzip curl
!mkdir -p /etc/apt/keyrings
!curl -fsSL https://dl.google.com/linux/linux_signing_key.pub | gpg --dearmor -o /etc/apt/keyrings/google-chrome.gpg
!echo "deb [arch=amd64 signed-by=/etc/apt/keyrings/google-chrome.gpg] http://dl.google.com/linux/chrome/deb/ stable main" | tee /etc/apt/sources.list.d/google-chrome.list > /dev/null
!apt-get update -y
!apt-get install -y google-chrome-stable
!pip -q install selenium pandas numpy

import os, re, json, time, glob, shutil, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

OPINET_DOWNLOAD_URL = "https://www.opinet.co.kr/user/opdown/opDownload.do"
MIN_STATION_DATE = pd.Timestamp("2008-04-15")

STATION_COLLECTION_DIR = Path(DATA_COLLECTION_PATH) / "gas_station_prices_by_region"
RAW_DIR = STATION_COLLECTION_DIR / "raw"
FINAL_DIR = STATION_COLLECTION_DIR / "final"
LOG_DIR = STATION_COLLECTION_DIR / "logs"
TMP_DIR = Path("/content/opinet_station_download_tmp")

PROCESSED_REGION_DIR = Path(PROCESSED_PATH) / "additional_data" / "gas_station_prices_by_region"

for p in [STATION_COLLECTION_DIR, RAW_DIR, FINAL_DIR, LOG_DIR, TMP_DIR, PROCESSED_REGION_DIR]:
    p.mkdir(parents=True, exist_ok=True)

ALL_REGIONS = [
    "서울", "부산", "대구", "인천", "광주", "대전", "울산", "세종",
    "경기", "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"
]

print("RAW_DIR              =", RAW_DIR)
print("FINAL_DIR            =", FINAL_DIR)
print("PROCESSED_REGION_DIR =", PROCESSED_REGION_DIR)

In [ ]:
# ============================================================
# 06-B 전국 개별 주유소/지역별 주유소 원자료 - 함수
# ============================================================
def _norm_text(x):
    return unicodedata.normalize("NFC", str(x)).strip()

def _safe_name(x):
    return re.sub(r'[\\/:*?"<>|]+', "_", str(x)).strip()

def _ymd(x):
    return pd.to_datetime(x).strftime("%Y%m%d")

def make_date_ranges(start_date, end_date=None):
    s = pd.to_datetime(start_date)
    e = pd.to_datetime(end_date or start_date)

    if s < MIN_STATION_DATE:
        s = MIN_STATION_DATE

    if e < s:
        raise ValueError("end_date가 start_date보다 빠릅니다.")

    ranges = []
    cur = s

    while cur <= e:
        y_end = min(pd.Timestamp(cur.year, 12, 31), e)
        ranges.append((str(cur.year), cur.strftime("%Y%m%d"), y_end.strftime("%Y%m%d")))
        cur = y_end + pd.Timedelta(days=1)

    return ranges

def debug_list_dir(path, limit=30):
    path = Path(path)
    print(f"[DIR] {path}")

    if not path.exists():
        print("  - not exists")
        return

    files = sorted(path.glob("*"), key=lambda p: p.stat().st_mtime if p.exists() else 0, reverse=True)
    print(f"  - count: {len(files)}")

    for p in files[:limit]:
        try:
            print(f"  - {p.name} | size={p.stat().st_size:,}")
        except Exception as e:
            print(f"  - {p.name} | stat error={e}")

def create_driver(download_dir, headless=True):
    opts = Options()
    opts.binary_location = "/usr/bin/google-chrome"

    opts.add_experimental_option("prefs", {
        "download.default_directory": str(download_dir),
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
    })

    if headless:
        opts.add_argument("--headless=new")

    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--window-size=1600,1200")
    opts.add_argument("--lang=ko-KR")

    driver = webdriver.Chrome(options=opts)
    driver.execute_cdp_cmd(
        "Page.setDownloadBehavior",
        {"behavior": "allow", "downloadPath": str(download_dir)}
    )
    return driver

def open_opinet_download_page(driver):
    driver.get(OPINET_DOWNLOAD_URL)
    wait = WebDriverWait(driver, 40)
    wait.until(EC.presence_of_element_located((By.ID, "span_start_date_picker")))
    wait.until(EC.presence_of_element_located((By.ID, "span_end_date_picker")))
    wait.until(EC.presence_of_element_located((By.ID, "sido")))
    wait.until(EC.presence_of_element_located((By.ID, "sigun")))

def set_daily_mode(driver):
    driver.execute_script("""
        document.getElementById('rdo3').checked = true;
        document.getElementById('rdo4').checked = true;
        document.getElementById('rdo3').dispatchEvent(new Event('change', {bubbles:true}));
        document.getElementById('rdo4').dispatchEvent(new Event('change', {bubbles:true}));
    """)

def set_region(driver, region):
    region = _norm_text(region)
    sel = Select(driver.find_element(By.ID, "sido"))

    for opt in sel.options:
        if _norm_text(opt.text) == region:
            sel.select_by_visible_text(opt.text.strip())
            WebDriverWait(driver, 30).until(
                lambda d: len(Select(d.find_element(By.ID, "sigun")).options) >= 1
            )
            time.sleep(1.5)
            return

    raise ValueError(f"시도 옵션 없음: {region}")

def get_sigun_list(driver):
    sel = Select(driver.find_element(By.ID, "sigun"))
    skip = {"", "시/군/구", "선택", "전체", "선택하세요", "선택하세요."}

    out = []
    for opt in sel.options:
        txt = opt.text.strip()
        val = (opt.get_attribute("value") or "").strip()
        if txt not in skip and val:
            out.append(txt)

    return out

def set_sigun(driver, sigun):
    sel = Select(driver.find_element(By.ID, "sigun"))

    for opt in sel.options:
        txt = opt.text.strip()
        val = (opt.get_attribute("value") or "").strip()
        if txt == sigun and val:
            sel.select_by_visible_text(txt)
            time.sleep(0.8)
            return

    raise ValueError(f"시군구 옵션 없음: {sigun}")

def set_dates(driver, sdt, edt):
    s = driver.find_element(By.ID, "span_start_date_picker")
    e = driver.find_element(By.ID, "span_end_date_picker")

    driver.execute_script("""
        arguments[0].removeAttribute('readonly');
        arguments[1].removeAttribute('readonly');
        arguments[0].value = arguments[2];
        arguments[1].value = arguments[3];

        arguments[0].dispatchEvent(new Event('input', {bubbles:true}));
        arguments[0].dispatchEvent(new Event('change', {bubbles:true}));
        arguments[1].dispatchEvent(new Event('input', {bubbles:true}));
        arguments[1].dispatchEvent(new Event('change', {bubbles:true}));
    """, s, e, sdt, edt)

    time.sleep(0.8)

def wait_new_file(download_dir, before, timeout=600):
    download_dir = Path(download_dir)
    start = time.time()
    last_sizes = {}

    while time.time() - start < timeout:
        now = set(glob.glob(str(download_dir / "*")))
        new_files = now - before

        csv_candidates = []
        for f in new_files:
            p = Path(f)

            if not p.is_file():
                continue
            if p.name.endswith(".crdownload"):
                continue
            if p.suffix.lower() != ".csv":
                continue
            if p.stat().st_size <= 0:
                continue

            csv_candidates.append(p)

        for p in sorted(csv_candidates, key=lambda x: x.stat().st_mtime, reverse=True):
            size = p.stat().st_size
            prev_size = last_sizes.get(str(p))

            if prev_size == size:
                return str(p)

            last_sizes[str(p)] = size

        time.sleep(1)

    print("[DOWNLOAD TIMEOUT] 완료된 CSV 파일을 찾지 못했습니다.")
    debug_list_dir(download_dir, limit=50)
    return None

def click_csv_download(driver):
    driver.execute_script("fn_Download(6);")

    try:
        WebDriverWait(driver, 10).until(EC.alert_is_present())
        alert = driver.switch_to.alert
        print("[confirm]", alert.text)
        alert.accept()
    except Exception:
        pass

def read_csv_auto(path):
    path = Path(path)

    expected_cols = {"번호", "지역", "상호", "주소", "기간", "상표", "셀프여부", "휘발유", "경유"}
    encodings = ["utf-8-sig", "utf-8", "cp949", "euc-kr"]

    candidates = []
    last_error = None

    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc)
            df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]

            col_score = len(expected_cols & set(df.columns))

            id_score = 0
            if "번호" in df.columns:
                id_score = int(
                    df["번호"].astype(str).str.strip().str.match(r"^A\d+$", na=False).sum()
                )

            date_score = 0
            if "기간" in df.columns:
                date_key = (
                    df["기간"].astype(str)
                    .str.replace(r"\.0$", "", regex=True)
                    .str.replace(r"\D", "", regex=True)
                    .str[:8]
                )
                date_score = int(
                    pd.to_datetime(date_key, format="%Y%m%d", errors="coerce").notna().sum()
                )

            candidates.append((col_score, id_score, date_score, enc, df))

        except Exception as e:
            last_error = e

    if not candidates:
        raise RuntimeError(f"CSV 읽기 실패: {path}\n원인: {last_error}")

    candidates.sort(key=lambda x: (x[0], x[1], x[2]), reverse=True)
    best_col_score, best_id_score, best_date_score, best_enc, best_df = candidates[0]

    if best_col_score < 5:
        raise RuntimeError(
            f"CSV 인코딩/컬럼 판별 실패: {path}\n"
            f"선택 인코딩: {best_enc}\n"
            f"컬럼 점수: {best_col_score}\n"
            f"컬럼: {list(best_df.columns)}"
        )

    best_df.attrs["source_encoding"] = best_enc
    return best_df

def clean_station_raw(df, path):
    df = df.copy()
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]

    required = ["번호", "지역", "상호", "주소", "기간", "상표", "셀프여부", "휘발유", "경유"]
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(f"필수 컬럼 누락 {missing}: {path}")

    df = df[required].copy()

    df["번호"] = df["번호"].astype(str).str.strip()
    df = df[df["번호"].str.match(r"^A\d+$", na=False)].copy()

    date_key = (
        df["기간"].astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"\D", "", regex=True)
        .str[:8]
    )
    df["date"] = pd.to_datetime(date_key, format="%Y%m%d", errors="coerce")
    df = df[df["date"].notna()].copy()

    for c in ["휘발유", "경유"]:
        df[c] = pd.to_numeric(
            df[c].astype(str).str.replace(",", "", regex=False).str.strip(),
            errors="coerce",
        )

    for c in ["지역", "상호", "주소", "상표", "셀프여부"]:
        s = df[c].astype("string").str.strip()
        df[c] = s.mask(s.isin(["", "nan", "None", "<NA>"]))

    df = df.rename(columns={
        "번호": "station_id",
        "지역": "region",
        "상호": "station_name",
        "주소": "address",
        "상표": "brand",
        "셀프여부": "is_self_raw",
        "휘발유": "price_gasoline",
        "경유": "price_diesel",
    })

    df["date_str"] = df["date"].dt.strftime("%Y-%m-%d")
    df["source_file"] = Path(path).name

    return df

def is_valid_station_raw_csv(path):
    path = Path(path)

    if not path.exists() or not path.is_file():
        return False

    try:
        with open(path, "rb") as f:
            head = f.read(8192).lower()
        if b"<html" in head or b"<!doctype html" in head or b"download.do" in head:
            return False
    except Exception:
        return False

    try:
        raw = read_csv_auto(path)
        cleaned = clean_station_raw(raw, path)
        return len(cleaned) > 0
    except Exception:
        return False

def delete_invalid_files_by_year(regions, years):
    years = [str(y) for y in years]
    removed = []

    for region in regions:
        region = _norm_text(region)
        raw_region_dir = RAW_DIR / region

        if not raw_region_dir.exists():
            continue

        for p in sorted(raw_region_dir.rglob("*.csv")):
            if not any(y in p.name for y in years):
                continue

            if not is_valid_station_raw_csv(p):
                size = p.stat().st_size if p.exists() else None
                print("[DELETE BAD FILE]", p, "size=", size)
                p.unlink()
                removed.append({"region": region, "file": str(p), "size": size})

    removed_df = pd.DataFrame(removed)
    print("[문제 파일 삭제 요약]")
    display(removed_df if len(removed_df) else pd.DataFrame([{"removed": 0}]))
    return removed_df

def collect_station_region(
    region,
    start_date,
    end_date=None,
    sigun_filter=None,
    headless=True,
    overwrite=False,
):
    region = _norm_text(region)

    raw_region_dir = RAW_DIR / region
    tmp_region_dir = TMP_DIR / region

    raw_region_dir.mkdir(parents=True, exist_ok=True)
    tmp_region_dir.mkdir(parents=True, exist_ok=True)

    driver = create_driver(tmp_region_dir, headless=headless)
    rows = []

    try:
        open_opinet_download_page(driver)
        set_daily_mode(driver)
        set_region(driver, region)

        siguns = get_sigun_list(driver)

        if sigun_filter:
            sigun_set = {_norm_text(s) for s in sigun_filter}
            siguns = [s for s in siguns if _norm_text(s) in sigun_set]

        print(f"[REGION] {region}")
        print(f"[SIGUN COUNT] {len(siguns)}")
        print(siguns)

        for sigun in siguns:
            for _, sdt, edt in make_date_ranges(start_date, end_date):
                fname = _safe_name(f"{region}_{sigun}_{sdt}_{edt}.csv")
                raw_path = raw_region_dir / fname

                if raw_path.exists() and raw_path.stat().st_size > 0 and not overwrite:
                    print("[SKIP]", raw_path.name)
                    rows.append({
                        "region": region,
                        "sigun": sigun,
                        "start": sdt,
                        "end": edt,
                        "status": "skip_exists",
                        "path": str(raw_path),
                    })
                    continue

                print(f"[DOWNLOAD] {region}/{sigun} {sdt}~{edt}")

                try:
                    set_region(driver, region)
                    set_sigun(driver, sigun)
                    set_dates(driver, sdt, edt)

                    before = set(glob.glob(str(tmp_region_dir / "*")))
                    click_csv_download(driver)

                    downloaded = wait_new_file(tmp_region_dir, before, timeout=600)
                    if not downloaded:
                        raise RuntimeError("download timeout or no completed csv")

                    downloaded = Path(downloaded)

                    if not is_valid_station_raw_csv(downloaded):
                        raise RuntimeError(f"downloaded file validation failed: {downloaded}")

                    if raw_path.exists():
                        raw_path.unlink()

                    shutil.move(str(downloaded), raw_path)

                    rows.append({
                        "region": region,
                        "sigun": sigun,
                        "start": sdt,
                        "end": edt,
                        "status": "ok",
                        "path": str(raw_path),
                        "size": raw_path.stat().st_size,
                    })

                    time.sleep(1)

                except Exception as e:
                    rows.append({
                        "region": region,
                        "sigun": sigun,
                        "start": sdt,
                        "end": edt,
                        "status": "fail",
                        "error": repr(e),
                        "path": str(raw_path),
                    })
                    print("[FAIL]", region, sigun, sdt, edt, repr(e))
                    debug_list_dir(tmp_region_dir, limit=20)

    finally:
        try:
            driver.quit()
        except Exception:
            pass

    manifest = pd.DataFrame(rows)
    manifest_path = LOG_DIR / f"download_manifest_{region}_{_ymd(start_date)}_{_ymd(end_date or start_date)}.csv"
    manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")

    print("[manifest]", manifest_path)
    return manifest

def build_price_matrix(df, value_col):
    temp = (
        df[["date", "station_id", value_col]]
        .sort_values(["date", "station_id"])
        .drop_duplicates(["date", "station_id"], keep="last")
    )

    wide = temp.pivot(index="date", columns="station_id", values=value_col)
    wide = wide.sort_index().sort_index(axis=1)

    if len(wide.index) > 0:
        wide = wide.reindex(pd.date_range(wide.index.min(), wide.index.max(), freq="D"))

    wide.index.name = "date"
    wide.index = pd.Index(wide.index.strftime("%Y-%m-%d"), name="date")

    return wide

def change_history(group_df, col):
    s = group_df[["date_str", col]].dropna().copy()

    if s.empty:
        return []

    s[col] = s[col].astype(str).str.strip()
    s = s[s[col] != ""]

    if s.empty:
        return []

    s = s.loc[s[col].shift() != s[col]]
    return s[["date_str", col]].values.tolist()

def build_metadata(df):
    meta = {}

    work = (
        df.sort_values(["station_id", "date"])
        .drop_duplicates(["station_id", "date"], keep="last")
        .copy()
    )

    for station_id, g in work.groupby("station_id", sort=True):
        meta[station_id] = {
            "region": change_history(g, "region"),
            "station_name": change_history(g, "station_name"),
            "address": change_history(g, "address"),
            "brand": change_history(g, "brand"),
            "is_self": change_history(g, "is_self_raw"),
        }

    return meta

def preprocess_station_region(region, pretty_json=False):
    region = _norm_text(region)

    raw_region_dir = RAW_DIR / region
    final_region_dir = FINAL_DIR / region
    processed_region_dir = PROCESSED_REGION_DIR / region

    final_region_dir.mkdir(parents=True, exist_ok=True)
    processed_region_dir.mkdir(parents=True, exist_ok=True)

    for old_log in [
        final_region_dir / "failed_files_log.csv",
        final_region_dir / "empty_cleaned_files_log.csv",
    ]:
        if old_log.exists():
            old_log.unlink()

    files = sorted(raw_region_dir.rglob("*.csv"))

    if not files:
        raise FileNotFoundError(f"raw CSV 없음: {raw_region_dir}")

    parts = []
    fails = []
    empty_files = []

    for i, path in enumerate(files, 1):
        try:
            raw = read_csv_auto(path)
            cleaned = clean_station_raw(raw, path)

            if len(cleaned) == 0:
                empty_files.append({
                    "file": str(path),
                    "raw_rows": len(raw),
                    "columns": ", ".join(map(str, raw.columns)),
                    "reason": "cleaned row count is zero",
                })
                continue

            parts.append(cleaned)

            if i % 50 == 0 or i == len(files):
                print(f"[READ] {region}: {i}/{len(files)} files")

        except Exception as e:
            fails.append({
                "file": str(path),
                "error": repr(e),
            })
            print("[PREPROCESS FAIL]", path.name, repr(e))

    if empty_files:
        empty_path = final_region_dir / "empty_cleaned_files_log.csv"
        pd.DataFrame(empty_files).to_csv(empty_path, index=False, encoding="utf-8-sig")
        print("[WARN] cleaned 0행 파일:", len(empty_files), empty_path)

    if fails:
        fail_path = final_region_dir / "failed_files_log.csv"
        pd.DataFrame(fails).to_csv(fail_path, index=False, encoding="utf-8-sig")
        print("[WARN] 처리 실패 파일:", len(fails), fail_path)

    if not parts:
        raise RuntimeError(f"[ERROR] {region}에서 유효한 주유소 가격 행이 없습니다: {raw_region_dir}")

    all_df = pd.concat(parts, ignore_index=True)
    all_df = (
        all_df
        .sort_values(["date", "station_id", "source_file"])
        .drop_duplicates(["station_id", "date"], keep="last")
        .copy()
    )

    if len(all_df) == 0 or all_df["date"].notna().sum() == 0:
        raise RuntimeError(f"[ERROR] {region} 병합 후 유효 date가 없습니다: {raw_region_dir}")

    gasoline = build_price_matrix(all_df, "price_gasoline")
    diesel = build_price_matrix(all_df, "price_diesel")
    metadata = build_metadata(all_df)

    outputs = {
        "gasoline": final_region_dir / "gasoline.csv",
        "diesel": final_region_dir / "diesel.csv",
        "metadata": final_region_dir / "metadata.json",
    }

    gasoline.to_csv(outputs["gasoline"], encoding="utf-8-sig")
    diesel.to_csv(outputs["diesel"], encoding="utf-8-sig")

    with open(outputs["metadata"], "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2 if pretty_json else None)

    for src in outputs.values():
        shutil.copy2(src, processed_region_dir / src.name)

    summary = pd.DataFrame([{
        "region": region,
        "raw_files": len(files),
        "used_files": len(parts),
        "empty_cleaned_files": len(empty_files),
        "failed_files": len(fails),
        "rows": len(all_df),
        "stations": all_df["station_id"].nunique(),
        "date_min": all_df["date"].min().strftime("%Y-%m-%d"),
        "date_max": all_df["date"].max().strftime("%Y-%m-%d"),
        "gasoline_shape": str(gasoline.shape),
        "diesel_shape": str(diesel.shape),
        "final_dir": str(final_region_dir),
        "processed_dir": str(processed_region_dir),
    }])

    summary_path = LOG_DIR / f"preprocess_summary_{region}.csv"
    summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

    print("[전처리 완료]", region)
    display(summary)

    return summary

In [ ]:
# ============================================================
# 06-B 전국 개별 주유소/지역별 주유소 원자료 - 실행
# ============================================================

# True: 오피넷에서 추가 수집
# False: 네가 raw 폴더에 넣어둔 기존 파일만 전처리
RUN_DOWNLOAD = RUN_OPINET_STATION_DOWNLOAD

# True: RAW_DIR/{지역} 전체 CSV 기준으로 gasoline/diesel/metadata 생성
RUN_PREPROCESS = RUN_OPINET_STATION_DOWNLOAD

# [2026]이면 2026년만 추가 수집
# None이면 FULL_START_DATE~FULL_END_DATE 전체기간 수집
YEARS_TO_RUN = [2026]
# YEARS_TO_RUN = None

FULL_START_DATE = "2008-01-01"
FULL_END_DATE = COLLECT_END_DATE

YEAR_END_DATE = {
    2026: COLLECT_END_DATE,
}

REGIONS_TO_RUN = ALL_REGIONS
# REGIONS_TO_RUN = ["서울"]
# REGIONS_TO_RUN = ["광주", "세종", "충남"]

HEADLESS = True
OVERWRITE_DOWNLOADS = False

# 2026 파일 중 HTML/깨진 CSV가 있으면 먼저 삭제
CLEAN_INVALID_2026_BEFORE_DOWNLOAD = True

# 큰 지역/문제 지역은 기간 분할 가능
# 일반적으로는 비워두면 됨
CUSTOM_PERIODS_BY_REGION = {
    # "충남": [("2026-01-01", "2026-03-31"), ("2026-04-01", "2026-06-11")],
}

# 특정 시군구만 받을 때 사용
SIGUN_FILTER_BY_REGION = {
    # "충남": ["공주시", "금산군"],
}

def make_collection_periods(years_to_run=None):
    if years_to_run is None:
        return [(FULL_START_DATE, FULL_END_DATE)]

    periods = []

    for year in years_to_run:
        year = int(year)

        start = pd.Timestamp(year=year, month=1, day=1)
        end = pd.Timestamp(year=year, month=12, day=31)

        if start < MIN_STATION_DATE:
            start = MIN_STATION_DATE

        if year in YEAR_END_DATE:
            end = pd.to_datetime(YEAR_END_DATE[year])

        periods.append((start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d")))

    return periods

base_periods = make_collection_periods(YEARS_TO_RUN)

if RUN_DOWNLOAD and CLEAN_INVALID_2026_BEFORE_DOWNLOAD and YEARS_TO_RUN is not None:
    delete_invalid_files_by_year(REGIONS_TO_RUN, YEARS_TO_RUN)

all_download_logs = []
all_preprocess_logs = []

for region in REGIONS_TO_RUN:
    region = _norm_text(region)

    print("=" * 100)
    print(f"[지역 시작] {region}")
    print(f"[raw 입력 폴더] {RAW_DIR / region}")

    periods = CUSTOM_PERIODS_BY_REGION.get(region, base_periods)
    sigun_filter = SIGUN_FILTER_BY_REGION.get(region)

    if RUN_DOWNLOAD:
        for start_date, end_date in periods:
            print(f"[수집 기간] {region}: {start_date} ~ {end_date}")

            dl = collect_station_region(
                region=region,
                start_date=start_date,
                end_date=end_date,
                sigun_filter=sigun_filter,
                headless=HEADLESS,
                overwrite=OVERWRITE_DOWNLOADS,
            )
            all_download_logs.append(dl)

    if RUN_PREPROCESS:
        pp = preprocess_station_region(region, pretty_json=False)
        all_preprocess_logs.append(pp)

download_log = pd.concat(all_download_logs, ignore_index=True) if all_download_logs else pd.DataFrame()
preprocess_log = pd.concat(all_preprocess_logs, ignore_index=True) if all_preprocess_logs else pd.DataFrame()

print("[다운로드 상태 요약]")
if len(download_log):
    display(download_log["status"].value_counts(dropna=False).reset_index())
    display(download_log)
else:
    print("다운로드 실행 안 함")

print("[전처리 요약]")
if len(preprocess_log):
    display(preprocess_log)
else:
    print("전처리 실행 안 함")

In [ ]:
# ============================================================
# 06-B 전국 개별 주유소/지역별 주유소 원자료 - 최종 검증
# ============================================================
EXPECTED_MIN_DATE = "2008-04-15"
EXPECTED_MAX_DATE_AT_LEAST = "2026-06-10"

def read_date_bounds_from_wide_csv(path):
    path = Path(path)

    if not path.exists():
        return {
            "exists": False,
            "rows": 0,
            "stations": 0,
            "date_min": None,
            "date_max": None,
        }

    header = pd.read_csv(path, nrows=0)
    stations = max(len(header.columns) - 1, 0)

    date_df = pd.read_csv(path, usecols=[0])
    date_col = date_df.columns[0]
    dates = pd.to_datetime(date_df[date_col], errors="coerce")

    return {
        "exists": True,
        "rows": len(date_df),
        "stations": stations,
        "date_min": dates.min().strftime("%Y-%m-%d") if dates.notna().any() else None,
        "date_max": dates.max().strftime("%Y-%m-%d") if dates.notna().any() else None,
    }

def validate_station_region_outputs(regions=None, check_raw_2026=True):
    regions = regions or ALL_REGIONS
    rows = []
    invalid_raw_rows = []

    for region in regions:
        region = _norm_text(region)

        final_region_dir = FINAL_DIR / region
        processed_region_dir = PROCESSED_REGION_DIR / region
        raw_region_dir = RAW_DIR / region

        gas_path = final_region_dir / "gasoline.csv"
        diesel_path = final_region_dir / "diesel.csv"
        meta_path = final_region_dir / "metadata.json"

        gas_info = read_date_bounds_from_wide_csv(gas_path)
        diesel_info = read_date_bounds_from_wide_csv(diesel_path)

        metadata_exists = meta_path.exists()
        metadata_station_count = None

        if metadata_exists:
            with open(meta_path, "r", encoding="utf-8") as f:
                metadata_station_count = len(json.load(f))

        summary_path = LOG_DIR / f"preprocess_summary_{region}.csv"
        summary_failed_files = None
        summary_empty_files = None

        if summary_path.exists():
            summary = pd.read_csv(summary_path)
            if len(summary):
                summary_failed_files = int(pd.to_numeric(summary.loc[0, "failed_files"], errors="coerce"))
                summary_empty_files = int(pd.to_numeric(summary.loc[0, "empty_cleaned_files"], errors="coerce"))

        if check_raw_2026 and raw_region_dir.exists():
            for p in sorted(raw_region_dir.rglob("*2026*.csv")):
                if not is_valid_station_raw_csv(p):
                    invalid_raw_rows.append({
                        "region": region,
                        "file": str(p),
                        "size": p.stat().st_size if p.exists() else None,
                    })

        row = {
            "region": region,

            "gasoline_exists": gas_info["exists"],
            "gasoline_rows": gas_info["rows"],
            "gasoline_station_cols": gas_info["stations"],
            "gasoline_date_min": gas_info["date_min"],
            "gasoline_date_max": gas_info["date_max"],

            "diesel_exists": diesel_info["exists"],
            "diesel_rows": diesel_info["rows"],
            "diesel_station_cols": diesel_info["stations"],
            "diesel_date_min": diesel_info["date_min"],
            "diesel_date_max": diesel_info["date_max"],

            "metadata_exists": metadata_exists,
            "metadata_station_count": metadata_station_count,

            "processed_gasoline_exists": (processed_region_dir / "gasoline.csv").exists(),
            "processed_diesel_exists": (processed_region_dir / "diesel.csv").exists(),
            "processed_metadata_exists": (processed_region_dir / "metadata.json").exists(),

            "summary_failed_files": summary_failed_files,
            "summary_empty_cleaned_files": summary_empty_files,
        }

        row["date_ok"] = (
            row["gasoline_date_min"] is not None
            and row["diesel_date_min"] is not None
            and row["gasoline_date_min"] <= EXPECTED_MIN_DATE
            and row["diesel_date_min"] <= EXPECTED_MIN_DATE
            and row["gasoline_date_max"] >= EXPECTED_MAX_DATE_AT_LEAST
            and row["diesel_date_max"] >= EXPECTED_MAX_DATE_AT_LEAST
        )

        row["files_ok"] = (
            row["gasoline_exists"]
            and row["diesel_exists"]
            and row["metadata_exists"]
            and row["processed_gasoline_exists"]
            and row["processed_diesel_exists"]
            and row["processed_metadata_exists"]
        )

        row["preprocess_ok"] = (
            row["summary_failed_files"] in [0, None]
        )

        row["station_count_ok"] = (
            row["gasoline_station_cols"] == row["diesel_station_cols"]
            and row["metadata_station_count"] == row["gasoline_station_cols"]
        )

        row["overall_ok"] = (
            row["files_ok"]
            and row["date_ok"]
            and row["preprocess_ok"]
            and row["station_count_ok"]
        )

        rows.append(row)

    result = pd.DataFrame(rows)
    invalid_raw = pd.DataFrame(invalid_raw_rows)

    print("[지역별 최종 검증]")
    display(result)

    print("[검증 실패 지역]")
    bad = result[~result["overall_ok"]].copy()
    display(bad if len(bad) else pd.DataFrame([{"message": "전체 지역 통과"}]))

    print("[2026 raw 문제 파일]")
    display(invalid_raw if len(invalid_raw) else pd.DataFrame([{"message": "문제 파일 없음"}]))

    result.to_csv(LOG_DIR / "station_region_final_validation.csv", index=False, encoding="utf-8-sig")
    if len(invalid_raw):
        invalid_raw.to_csv(LOG_DIR / "station_region_invalid_2026_raw_files.csv", index=False, encoding="utf-8-sig")

    return result, invalid_raw

validation_result, invalid_2026_raw = validate_station_region_outputs(
    regions=ALL_REGIONS,
    check_raw_2026=True,
)

### 현재 레포 outputs 구조로 최종 산출물 정리

수집 셀은 `{dataset}/raw`, `{dataset}/final` snapshot도 남깁니다. 아래 셀은 01 전처리가 읽는 `data-analysis/00_data_collection/outputs/{dataset}/` 바로 아래로 최신 final 파일을 복사합니다.


In [ ]:
# ============================================================
# outputs/{dataset}/ 최신 final 파일 평탄화
# ============================================================
from pathlib import Path
import shutil

def _latest_file(base, pattern):
    base = Path(base)
    if not base.exists():
        return None
    files = [p for p in base.glob(pattern) if p.is_file()]
    return sorted(files, key=lambda p: (p.stat().st_mtime, str(p)), reverse=True)[0] if files else None

def _copy_latest_final(dataset, pattern):
    dataset_dir = Path(DATA_COLLECTION_PATH) / dataset
    src = _latest_file(dataset_dir / "final", pattern)
    if src is None:
        print(f"[SKIP] {dataset}/{pattern}: final snapshot 없음")
        return None
    dst = dataset_dir / src.name
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f"[COPY] {src} -> {dst}")
    return dst

FLATTEN_SPECS = [
    ("crude", "crude_*.csv"),
    ("retail_avg", "retail_avg_*.csv"),
    ("brand_price", "brand_gasoline_*.csv"),
    ("brand_price", "brand_diesel_*.csv"),
    ("fx_usdkrw", "fx_usdkrw_*.csv"),
    ("intl_products", "intl_products_*.csv"),
    ("intl_products", "intl_product_diesel(0.001)_*.csv"),
    ("fuel_tax_trend", "gasoline_tax_trend_*.xls"),
    ("fuel_tax_trend", "diesel_tax_trend_*.xls"),
    ("refinery_weekly_supply", "refinery_weekly_supply_prices_by_product_*.csv"),
]

flattened_outputs = []
for dataset, pattern in FLATTEN_SPECS:
    out = _copy_latest_final(dataset, pattern)
    if out is not None:
        flattened_outputs.append(str(out))

print(f"[DONE] flattened_output_count={len(flattened_outputs)}")
